In [1]:
import pandas as pd
import numpy as np

# 1.市场交易数据（月度）

In [2]:
# 产出变量：`Excess_Return`, `Size`, `Msmvttl`, `Mnshrtrd`, `MOM`, `REV`, `VOL`, `ILLIQ`, `TURN`

## 1.1 无风险利率清洗

In [3]:
rf_raw = pd.read_csv('TRD_Nrrate.csv')

# 将Clsdt转为日期格式
rf_raw['Clsdt'] = pd.to_datetime(rf_raw['Clsdt'])

# 提取年月
rf_raw['year_month'] = rf_raw['Clsdt'].dt.to_period('M')

# 每月只取第一条记录（所有天的值都一样）
rf_monthly = rf_raw.groupby('year_month')['Nrrmtdt'].first().reset_index()
rf_monthly.columns = ['year_month', 'rf']
# 关键：将百分数形式转换为小数形式
rf_monthly['rf'] = rf_monthly['rf'] / 100

# 统一数据类型，为合并做准备
rf_monthly['year_month'] = rf_monthly['year_month'].astype(str)

print("无风险利率处理完成，前5行预览：")
print(rf_monthly.head())

无风险利率处理完成，前5行预览：
  year_month        rf
0    2011-01  0.002263
1    2011-02  0.002263
2    2011-03  0.002466
3    2011-04  0.002466
4    2011-05  0.002669


## 1.2 个股回报率清洗

In [4]:
# 1. 读取个股收益率数据
# 在读取时就只保留有用的列，节省内存
cols_to_use = ['Stkcd', 'Trdmnt', 'Mnshrtrd', 'Msmvosd', 'Mretwd', 'Msmvttl', 'Mnvaltrd']
df = pd.read_csv('TRD_Mnth.csv', usecols=cols_to_use)

# 2. 日期转换与标准化
df['Trdmnt'] = pd.to_datetime(df['Trdmnt'], format='%y-%b')
df['year_month'] = df['Trdmnt'].dt.to_period('M').astype(str)

# 3. 剔除B股
df = df[~df['Stkcd'].astype(str).str.startswith('900')]  # 沪市B股
df = df[~df['Stkcd'].astype(str).str.startswith('200')]  # 深市B股

## 1.3 合并并计算超额收益率（`Excess_Return`）与规模（`Size`）

In [5]:
# 以 year_month 为键，把 rf 列匹配到每只股票上
df = df.merge(rf_monthly, on='year_month', how='left')

# 检查合并是否成功
missing_rf = df['rf'].isnull().sum()
if missing_rf > 0:
    print(f"警告：有 {missing_rf} 行无法匹配到无风险利率，请检查时间区间是否一致。")
else:
    print("无风险利率匹配成功，无缺失值。")

无风险利率匹配成功，无缺失值。


In [6]:
# 处理单位与计算变量
# 1. 单位转换：流通市值（千元 → 元）后取自然对数
df['Size'] = np.log(df['Msmvosd'] * 1000)
df['Excess_Return'] = df['Mretwd'] - df['rf']

# 2. 计算超额收益率
df['Excess_Return'] = df['Mretwd'] - df['rf']

# 快速预览
print("\n最终数据前5行：")
print(df[['Stkcd', 'Trdmnt', 'Mretwd', 'rf', 'Excess_Return', 'Size', 'Msmvttl']].head())

print(f"\n总观测数：{len(df):,}")
print(f"股票数量：{df['Stkcd'].nunique()}")
print(f"时间跨度：{df['year_month'].min()} 至 {df['year_month'].max()}")


最终数据前5行：
   Stkcd     Trdmnt    Mretwd        rf  Excess_Return       Size      Msmvttl
0      1 2011-01-01 -0.030399  0.002263      -0.032662  24.584901  53355560.70
1      1 2011-02-01  0.040496  0.002263       0.038233  24.624599  55516269.23
2      1 2011-03-01  0.009416  0.002466       0.006950  24.633971  56039021.29
3      1 2011-04-01  0.131841  0.002466       0.129375  24.757817  63427250.47
4      1 2011-05-01 -0.029670  0.002669      -0.032339  24.727697  61545343.04

总观测数：646,374
股票数量：5712
时间跨度：2011-01 至 2025-12


## 1.4 计算自算市场变量：`MOM`, `REV`, `VOL`, `ILLIQ`

In [7]:
# MOM = 过去12个月累计超额收益（跳过最近一个月）
# REV = 上个月超额收益
# VOL = 过去12个月超额收益标准差
# ILLIQ = |月收益| / 交易金额，Amihud (2002) 非流动性指标
# 运算前提：df 已按 Stkcd + Trdmnt 排序

In [8]:
# 确保数据排序正确，滚动计算基础
df = df.sort_values(['Stkcd', 'Trdmnt']).reset_index(drop=True)

In [9]:
# MOM：过去12个月累计收益（t-12至t-2，不含当月）
df['MOM'] = df.groupby('Stkcd')['Excess_Return'].transform(
    lambda x: x.shift(1).rolling(11, min_periods=8).sum()  # shift(1)排除当月，rolling(11)即为过去11个月（t-12到t-2）
)

In [10]:
# REV：上月收益
df['REV'] = df.groupby('Stkcd')['Excess_Return'].shift(1)

In [11]:
# VOL：过去12个月超额收益标准差
df['VOL'] = df.groupby('Stkcd')['Excess_Return'].transform(
    lambda x: x.shift(1).rolling(12, min_periods=8).std()
)

In [12]:
# ILLIQ: Amihud非流动性指标 = |Excess_Return| / 交易金额（元）
# Mnvaltrd 单位为元
df['ILLIQ'] = np.abs(df['Excess_Return']) / df['Mnvaltrd']

In [13]:
print("1.4 自算市场变量完成")
print(df[['Stkcd','Trdmnt','Excess_Return','MOM','REV','VOL','ILLIQ']].head())

print(f"\n最终主表 df 样本量：{len(df):,}，股票数：{df['Stkcd'].nunique()}")
print(f"时间跨度：{df['year_month'].min()} 至 {df['year_month'].max()}")
print("变量列表：", df.columns.tolist())

1.4 自算市场变量完成
   Stkcd     Trdmnt  Excess_Return  MOM       REV  VOL         ILLIQ
0      1 2011-01-01      -0.032662  NaN       NaN  NaN  4.217270e-12
1      1 2011-02-01       0.038233  NaN -0.032662  NaN  5.895531e-12
2      1 2011-03-01       0.006950  NaN  0.038233  NaN  5.513088e-13
3      1 2011-04-01       0.129375  NaN  0.006950  NaN  7.194451e-12
4      1 2011-05-01      -0.032339  NaN  0.129375  NaN  4.259394e-12

最终主表 df 样本量：646,374，股票数：5712
时间跨度：2011-01 至 2025-12
变量列表： ['Stkcd', 'Trdmnt', 'Mnshrtrd', 'Msmvosd', 'Mretwd', 'Msmvttl', 'Mnvaltrd', 'year_month', 'rf', 'Size', 'Excess_Return', 'MOM', 'REV', 'VOL', 'ILLIQ']


In [14]:
print(df[['MOM','REV','VOL','ILLIQ']].isnull().mean()) # 确认缺失主要集中在每只股票的早期月份，符合预期

MOM      0.075744
REV      0.014507
VOL      0.075744
ILLIQ    0.005698
dtype: float64


## 1.5 合并换手率 `TURN`

In [15]:
# 换手率文件是 Wind 导出的，列名包含日期，需要逆透视
import glob
import re

In [16]:
# 获取所有换手率文件
turnover_files = glob.glob('*月换手率.csv')
print(f"找到 {len(turnover_files)} 个换手率文件")

找到 15 个换手率文件


In [18]:
all_turnover = []

for file in turnover_files:
    # 读取文件
    temp = pd.read_csv(file, encoding='utf-8-sig')
    
    # 动态检测证券代码列
    code_cols = [col for col in temp.columns if '证券' in col or '代码' in col or 'Stkcd' in col]
    if not code_cols:
        # 如果没有找到，打印列名帮助调试
        print(f"  警告：{file} 未找到证券代码列，列名如下：")
        print(temp.columns.tolist())
        continue
    
    code_col = code_cols[0]  # 取第一个匹配的列
    print(f"  {file}: 使用列名 '{code_col}'")
    
    # 清洗证券代码
    temp['Stkcd'] = temp[code_col].astype(str).str.replace(r'\.\w+$', '', regex=True).str.strip()
    temp['Stkcd'] = temp['Stkcd'].astype(int).astype(str)
    
    # 找月份列
    month_cols = [col for col in temp.columns if '[交易日期]' in col]
    if not month_cols:
        month_cols = [col for col in temp.columns if re.search(r'\d{4}-\d{1,2}-\d{1,2}', col)]
    
    print(f"    找到 {len(month_cols)} 个月份列")
    
    # 逆透视
    melted = temp.melt(id_vars=['Stkcd'], value_vars=month_cols, 
                       var_name='month_label', value_name='TURN_raw')
    
    # 提取年月
    melted['year_month'] = melted['month_label'].str.extract(r'(\d{4}-\d{1,2})')[0]
    melted['year_month'] = pd.to_datetime(melted['year_month']).dt.to_period('M').astype(str)
    
    # 转数值并计算换手率
    melted['TURN_raw'] = pd.to_numeric(melted['TURN_raw'], errors='coerce')
    melted['TURN'] = melted['TURN_raw'] / 100.0
    
    # 清理
    melted = melted[['Stkcd', 'year_month', 'TURN']]
    melted = melted.dropna(subset=['TURN'])
    
    all_turnover.append(melted)
    print(f"    处理完成，共 {len(melted)} 条记录")

  2011月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 23568 条记录
  2012月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 26047 条记录
  2013月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 26547 条记录
  2014月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 26602 条记录
  2015月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 27739 条记录
  2016月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 30313 条记录
  2017月换手率.csv: 使用列名 '证券代码'
    找到 12 个月份列
    处理完成，共 35205 条记录
  2018月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 38619 条记录
  2019月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 41252 条记录
  2020月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 44865 条记录
  2021月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 51122 条记录
  2022月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 56553 条记录
  2023月换手率.csv: 使用列名 '证券代码↑'
    找到 12 个月份列
    处理完成，共 61312 条记录
  2024月换手率.csv: 使用列名 '证券代码'
    找到 12 个月份列
    处理完成，共 63584 条记录
  2025月换手率.csv: 使用列名 '证券代码'
    找到 12 个月份列
    处理完成，共 64842 条记录


In [19]:
# 合并所有年份的换手率
turnover_all = pd.concat(all_turnover, ignore_index=True)

# 去重
turnover_all = turnover_all.drop_duplicates(subset=['Stkcd', 'year_month'])

print(f"\n换手率面板合并完成！共 {len(turnover_all)} 条记录")
print(f"时间跨度：{turnover_all['year_month'].min()} 至 {turnover_all['year_month'].max()}")
print(turnover_all.head(10))

# 合并到主表 df
# 确保 df 中的 Stkcd 也是字符串形式
df['Stkcd_str'] = df['Stkcd'].astype(int).astype(str)

# 左连接
df = df.merge(turnover_all, left_on=['Stkcd_str', 'year_month'], 
              right_on=['Stkcd', 'year_month'], how='left')

# 清理列
df.drop(columns=['Stkcd_str', 'Stkcd_y'], errors='ignore', inplace=True)
df.rename(columns={'Stkcd_x': 'Stkcd'}, inplace=True, errors='ignore')

print(f"\n合并后 df 样本量：{len(df):,}")
print(f"换手率 TURN 缺失比例：{df['TURN'].isnull().mean():.2%}")


换手率面板合并完成！共 618170 条记录
时间跨度：2011-01 至 2025-12
  Stkcd year_month      TURN
0     1    2011-12  0.121352
1     2    2011-12  0.114939
2     4    2011-12  0.223392
3     6    2011-12  0.064392
4     7    2011-12  0.121838
5     8    2011-12  0.206629
6     9    2011-12  0.479103
7    11    2011-12  0.835984
8    12    2011-12  0.460243
9    14    2011-12  0.098452

合并后 df 样本量：646,374
换手率 TURN 缺失比例：4.67%


# 2.财务特征数据（季度→月度）

In [20]:
# 产出变量：`ROA`, `ROE`, `GP`, `LEV`, `ATO`, `BM`, `CFO`, `TotalAsset` → `IA`, `ACC`, `EP`

## 2.1 ST历史状态清洗

In [21]:
st_raw = pd.read_csv('SPT_Trdchg.csv')

# 只保留关键列
st = st_raw[['Stkcd', 'Chgtype', 'Execudt']].copy()
st['Execudt'] = pd.to_datetime(st['Execudt'])

# B = ST, D = *ST, C = PT, S = 暂停上市, T = 退市整理期
st['new_status'] = st['Chgtype'].str[-1]  # 取第二位
st['is_ST_start'] = st['new_status'].isin(['B', 'D', 'C', 'S', 'T'])

# 按股票分组，为每段状态找结束日期
st = st.sort_values(['Stkcd', 'Execudt'])
st['end_date'] = st.groupby('Stkcd')['Execudt'].shift(-1)

# 只保留ST状态的行
st_periods = st[st['is_ST_start']].copy()
st_periods = st_periods.rename(columns={'Execudt': 'start_date'})
st_periods = st_periods[['Stkcd', 'start_date', 'end_date', 'Chgtype']].reset_index(drop=True)

# 如果最后一段ST没有结束日期（至今未脱帽），则用远期日期填充
st_periods['end_date'] = st_periods['end_date'].fillna(pd.Timestamp('2026-01-01'))

print(f"ST历史清洗完成，共 {len(st_periods)} 段ST记录")
print(st_periods.head(10))

ST历史清洗完成，共 1567 段ST记录
   Stkcd start_date   end_date Chgtype
0      4 2022-05-06 2023-06-28      AB
1      4 2025-04-30 2026-01-01      AD
2      5 2021-05-06 2024-04-26      AB
3      7 2021-04-30 2022-07-01      AD
4      7 2023-05-05 2024-07-02      AD
5     10 2019-04-26 2020-07-22      AD
6     17 2020-04-29 2021-06-22      AD
7     18 2013-04-24 2014-03-13      BD
8     18 2019-05-06 2019-11-25      AD
9     18 2019-11-25 2020-01-07      DT


## 2.2 财务数据季度总表合并（`fin`）

In [22]:
def read_fin(file, cols, has_typrep=True):
    """
    读取财务表格，保留合并报表(A)的记录
    """
    df = pd.read_csv(file)
    if has_typrep and 'Typrep' in df.columns:
        df = df[df['Typrep'] == 'A'].copy()
    return df[cols]

# 1. FI_T5: 营业毛利率(GP) + 总资产净利润率(ROA) + 净资产收益率(ROE)
fin1 = read_fin('FI_T5.CSV', ['Stkcd', 'Accper', 'F050201B', 'F050501B', 'F053301B'])
fin1.columns = ['Stkcd', 'Accper', 'ROA', 'ROE', 'GP']

# 2. FI_T1: 资产负债率(LEV)
fin2 = read_fin('FI_T1.CSV', ['Stkcd', 'Accper', 'F011201A'])
fin2.columns = ['Stkcd', 'Accper', 'LEV']

# 3. FI_T4: 总资产周转率(ATO)
fin3 = read_fin('FI_T4.CSV', ['Stkcd', 'Accper', 'F041701B'])
fin3.columns = ['Stkcd', 'Accper', 'ATO']

# 4. FI_T10: 账面市值比(BM) —— 此表无Typrep字段，has_typrep=False
fin4 = read_fin('FI_T10.CSV', ['Stkcd', 'Accper', 'F101001A'], has_typrep=False)
fin4.columns = ['Stkcd', 'Accper', 'BM']

# 5. FS_Comscfd: 经营活动现金流量净额(CFO) —— 用于计算应计利润(ACC)
fin5 = read_fin('FS_Comscfd.csv', ['Stkcd', 'Accper', 'C001000000'])
fin5.columns = ['Stkcd', 'Accper', 'CFO']

# 6. FS_Combas: 资产总计(TotalAsset) —— 用于计算总资产增长率(IA)和ACC的分母
fin6 = read_fin('FS_Combas.csv', ['Stkcd', 'Accper', 'A001000000'])
fin6.columns = ['Stkcd', 'Accper', 'TotalAsset']

# 逐一合并
fin = fin1.merge(fin2, on=['Stkcd', 'Accper'], how='outer')
fin = fin.merge(fin3, on=['Stkcd', 'Accper'], how='outer')
fin = fin.merge(fin4, on=['Stkcd', 'Accper'], how='outer')
fin = fin.merge(fin5, on=['Stkcd', 'Accper'], how='outer')
fin = fin.merge(fin6, on=['Stkcd', 'Accper'], how='outer')

# 日期处理
fin['Accper'] = pd.to_datetime(fin['Accper'])
fin = fin.sort_values(['Stkcd', 'Accper']).reset_index(drop=True)

print(f"财务数据总表：{len(fin)} 行，{fin['Stkcd'].nunique()} 只股票")
print(f"时间跨度：{fin['Accper'].min()} 至 {fin['Accper'].max()}")
print("\n字段列表：", fin.columns.tolist())
print("\n前5行预览：")
print(fin.head())

财务数据总表：308088 行，5890 只股票
时间跨度：2011-01-01 00:00:00 至 2025-12-31 00:00:00

字段列表： ['Stkcd', 'Accper', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'BM', 'CFO', 'TotalAsset']

前5行预览：
   Stkcd     Accper       ROA       ROE  GP       LEV       ATO        BM  \
0      1 2011-01-01       NaN       NaN NaN       NaN       NaN       NaN   
1      1 2011-03-31  0.002975  0.066820 NaN  0.955483  0.006956  0.978315   
2      1 2011-06-30  0.005553  0.124022 NaN  0.955223  0.013666  0.978174   
3      1 2011-09-30  0.006416  0.108041 NaN  0.940618  0.016737  0.994760   
4      1 2011-12-31  0.008258  0.137840 NaN  0.940087  0.023119  0.997840   

            CFO    TotalAsset  
0  2.204572e+10  7.272071e+11  
1  2.134655e+10  8.075634e+11  
2  3.157390e+10  8.520570e+11  
3  1.786163e+10  1.207212e+12  
4 -1.443937e+10  1.258177e+12  


In [23]:
# 保留标准季度报表日
fin = fin[fin['Accper'].dt.month.isin([3, 6, 9, 12])].copy()

# 再次确认
print(f"过滤后财务数据总表：{len(fin)} 行，{fin['Stkcd'].nunique()} 只股票")
print(f"时间跨度：{fin['Accper'].min()} 至 {fin['Accper'].max()}")
print(fin['Accper'].dt.month.value_counts().sort_index())  # 应只有 3,6,9,12

过滤后财务数据总表：247803 行，5890 只股票
时间跨度：2011-03-31 00:00:00 至 2025-12-31 00:00:00
Accper
3     57831
6     60723
9     59545
12    69704
Name: count, dtype: int64


## 2.3 季度→月度填充（按披露截止日规则）

In [24]:
# 将仅包含 3/6/9/12 月四条记录的季度财务表 fin按报告披露截止日规则，填充到 1–12 月。
# 前提：fin 的 Accper 列已经是 datetime 类型。

### 2.3.1 准备完整的股票 × 月份 面板

In [25]:
# 从主表 df 获取所有股票代码和所有月份的排列
all_stocks = df[['Stkcd']].drop_duplicates()
all_months = df[['year_month', 'Trdmnt']].drop_duplicates().sort_values('Trdmnt')

In [26]:
# 构造完整的月份序列（每月第一天，便于逻辑判断）
all_months['month_dt'] = all_months['Trdmnt'].values

In [27]:
# 生成所有股票 × 所有月份的面板
panel = all_stocks.assign(key=1).merge(all_months.assign(key=1), on='key').drop('key', axis=1)
panel = panel.sort_values(['Stkcd', 'Trdmnt']).reset_index(drop=True)

### 2.3.2 处理财务数据，准备季度标识

In [28]:
fin_q = fin.copy()
fin_q['year'] = fin_q['Accper'].dt.year
fin_q['qtr']  = fin_q['Accper'].dt.quarter

In [29]:
# 为每一季度创建向后兼容的多条记录
# 方法：将 Accper 列按季度扩展，直接与面板匹配。

# 关键：定义每个月份（1-12）应使用的财务报告期
def map_month_to_report(month_dt):
    """
    输入：某一月份的第一天 datetime
    返回：应使用的会计报表日（Accper）的月份标记 (年, 月) 
    规则：
        Jan–Apr → 上一年的 Q3 报告（因为披露截止在4月30日前用去年年报，这里按原论文规则仔细实现）
        1月1日至4月30日：使用上一年Q3的报告（即报告日为去年9月30日）；
        5月1日至8月31日：使用当年Q1的报告（报告日为当年3月31日）；
        9月1日至10月31日：使用当年Q2的报告（报告日为当年6月30日）；
        11月1日至12月31日：使用当年Q3的报告（报告日为当年9月30日）。
    """
    m = month_dt.month
    y = month_dt.year
    
    if m <= 4:
        return pd.Timestamp(year=y-1, month=9, day=30)
    elif m <= 8:
        return pd.Timestamp(year=y, month=3, day=31)
    elif m <= 10:
        return pd.Timestamp(year=y, month=6, day=30)
    else:
        return pd.Timestamp(year=y, month=9, day=30)

In [30]:
# 对面板中的每个月份，计算对应的报告日期
panel['report_date'] = panel['month_dt'].apply(map_month_to_report)

### 2.3.3 合并财务数据

In [31]:
fin_monthly = panel.merge(fin_q, left_on=['Stkcd', 'report_date'], 
                          right_on=['Stkcd', 'Accper'], how='left')

In [32]:
# 只保留需要的列，去除冗余
keep_cols = ['Stkcd', 'year_month', 'Trdmnt', 'ROA', 'ROE', 'GP', 
             'LEV', 'ATO', 'BM', 'CFO', 'TotalAsset']
fin_monthly = fin_monthly[keep_cols]

In [33]:
print(f"月度财务数据表：{len(fin_monthly)} 行")
print(f"时间跨度：{fin_monthly['Trdmnt'].min()} 至 {fin_monthly['Trdmnt'].max()}")
print("变量列表：", fin_monthly.columns.tolist())
print("\n前 5 行预览：")
print(fin_monthly.head(10))

月度财务数据表：1028160 行
时间跨度：2011-01-01 00:00:00 至 2025-12-01 00:00:00
变量列表： ['Stkcd', 'year_month', 'Trdmnt', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'BM', 'CFO', 'TotalAsset']

前 5 行预览：
   Stkcd year_month     Trdmnt       ROA       ROE  GP       LEV       ATO  \
0      1    2011-01 2011-01-01       NaN       NaN NaN       NaN       NaN   
1      1    2011-02 2011-02-01       NaN       NaN NaN       NaN       NaN   
2      1    2011-03 2011-03-01       NaN       NaN NaN       NaN       NaN   
3      1    2011-04 2011-04-01       NaN       NaN NaN       NaN       NaN   
4      1    2011-05 2011-05-01  0.002975  0.066820 NaN  0.955483  0.006956   
5      1    2011-06 2011-06-01  0.002975  0.066820 NaN  0.955483  0.006956   
6      1    2011-07 2011-07-01  0.002975  0.066820 NaN  0.955483  0.006956   
7      1    2011-08 2011-08-01  0.002975  0.066820 NaN  0.955483  0.006956   
8      1    2011-09 2011-09-01  0.005553  0.124022 NaN  0.955223  0.013666   
9      1    2011-10 2011-10-01  0.005553  0.

## 2.4 计算自算财务变量：`IA`（总资产增长率）、`ACC`（应计利润）、`EP`（市盈率倒数）

In [34]:
# IA   = 总资产增长率 = (期末总资产 - 期初总资产) / 期初总资产
# ACC  = 应计利润 = (净利润 - 经营活动现金流) / 总资产
# EP   = 市盈率倒数 = 净利润 / 总市值
# 前提：fin_monthly 已生成，df 已包含 Msmvttl
# 说明：此处净利润 = ROE × 净资产，净资产 = 总资产 × (1 - LEV)

### 2.4.1 计算 IA（总资产同比增长率）

In [35]:
fin_monthly = fin_monthly.sort_values(['Stkcd', 'Trdmnt']).reset_index(drop=True)

In [36]:
# 同一季度报告的总资产，在填充后多个月份应一致。
# 为了计算年度增长率，使用每只股票在每个报告日的总资产
# 然后按股票排序后，与 12 个月前的总资产对比。
# 更精准的做法：在原始季度表 fin 中先计算再填充；
# 简便做法：在 fin_monthly 中直接用 shift(12)
fin_monthly['TotalAsset_lag12'] = fin_monthly.groupby('Stkcd')['TotalAsset'].shift(12)
fin_monthly['IA'] = (fin_monthly['TotalAsset'] - fin_monthly['TotalAsset_lag12']) / fin_monthly['TotalAsset_lag12'].abs()

### 2.4.2 计算 ACC

In [37]:
# ACC = (净利润 - CFO) / 总资产
# 净利润 = ROE × 净资产，净资产 = 总资产 × (1 - LEV)
fin_monthly['NetProfit'] = fin_monthly['ROE'] * fin_monthly['TotalAsset'] * (1 - fin_monthly['LEV'])
fin_monthly['ACC'] = (fin_monthly['NetProfit'] - fin_monthly['CFO']) / fin_monthly['TotalAsset']

### 2.4.3 计算 EP

In [38]:
# EP = 净利润 / 总市值
# 需要从 df 合并总市值 Msmvttl（单位千元，需转为元）
df_market = df[['Stkcd', 'year_month', 'Msmvttl']].copy()
df_market['Msmvttl_yuan'] = df_market['Msmvttl'] * 1000  # 千元 → 元
fin_monthly = fin_monthly.merge(df_market, on=['Stkcd', 'year_month'], how='left')
fin_monthly['EP'] = fin_monthly['NetProfit'] / fin_monthly['Msmvttl_yuan']

### 2.4.4 清理中间变量

In [39]:
fin_monthly = fin_monthly.drop(columns=['TotalAsset_lag12', 'NetProfit', 'Msmvttl', 'Msmvttl_yuan'], errors='ignore')

In [40]:
print("自算财务变量计算完成。")
print(f"IA  描述：{fin_monthly['IA'].describe()}")
print(f"ACC 描述：{fin_monthly['ACC'].describe()}")
print(f"EP  描述：{fin_monthly['EP'].describe()}")
print(f"\n当前变量列表：{fin_monthly.columns.tolist()}")

自算财务变量计算完成。
IA  描述：count    577748.000000
mean          0.415205
std          25.071797
min          -0.993357
25%          -0.006407
50%           0.070742
75%           0.179598
max        5796.560827
Name: IA, dtype: float64
ACC 描述：count    634766.000000
mean          0.019034
std           1.491960
min         -17.990367
25%          -0.013872
50%           0.009023
75%           0.038174
max         483.721092
Name: ACC, dtype: float64
EP  描述：count    622979.000000
mean          0.015283
std           0.079667
min         -14.762801
25%           0.002274
50%           0.009940
75%           0.023686
max           5.219501
Name: EP, dtype: float64

当前变量列表：['Stkcd', 'year_month', 'Trdmnt', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'BM', 'CFO', 'TotalAsset', 'IA', 'ACC', 'EP']


# 3.机制检验与样本筛选

## 3.1 指数成分股清洗（沪深300 & 中证500）

In [41]:
# 将多个 IDX_Chgsmp CSV 文件合并，提取沪深300(000300)和中证500(000905)的成分股变更记录，生成月度级别的指数成员标识。
# 输出：idx_monthly，含 Stkcd, year_month, in_HS300, in_ZZ500

### 3.1.1 读取并合并所有 IDX_Chgsmp 文件

In [42]:
# 手动列出Jupyter Notebook里所有指数成分股文件
file_list = [
    'IDX_Chgsmp(11-13).csv',
    'IDX_Chgsmp(14-16).csv',
    'IDX_Chgsmp(17-19).csv',
    'IDX_Chgsmp(20-22)(1).csv',
    'IDX_Chgsmp(20-22)(2).csv',
    'IDX_Chgsmp(20-22)(3).csv',
    'IDX_Chgsmp(23-25)(1).csv',
    'IDX_Chgsmp(23-25)(2).csv',
    'IDX_Chgsmp(23-25)(3).csv',
    'IDX_Chgsmp(23-25)(4).csv',
    'IDX_Chgsmp(23-25)(5).csv',
    'IDX_Chgsmp(23-25)(6).csv',
    'IDX_Chgsmp(23-25)(7).csv'
]

print(f"共读取 {len(file_list)} 个指数成分股文件")

共读取 13 个指数成分股文件


In [43]:
df_idx_raw_list = []
for f in file_list:
    temp = pd.read_csv(f, low_memory=False)   # low_memory=False 消除 DtypeWarning
    df_idx_raw_list.append(temp)
    print(f"  {f} → {len(temp)} 行")

idx_raw = pd.concat(df_idx_raw_list, ignore_index=True)

  IDX_Chgsmp(11-13).csv → 242237 行
  IDX_Chgsmp(14-16).csv → 365374 行
  IDX_Chgsmp(17-19).csv → 772321 行
  IDX_Chgsmp(20-22)(1).csv → 1000000 行
  IDX_Chgsmp(20-22)(2).csv → 1000000 行
  IDX_Chgsmp(20-22)(3).csv → 39218 行
  IDX_Chgsmp(23-25)(1).csv → 1000000 行
  IDX_Chgsmp(23-25)(2).csv → 1000000 行
  IDX_Chgsmp(23-25)(3).csv → 1000000 行
  IDX_Chgsmp(23-25)(4).csv → 1000000 行
  IDX_Chgsmp(23-25)(5).csv → 1000000 行
  IDX_Chgsmp(23-25)(6).csv → 1000000 行
  IDX_Chgsmp(23-25)(7).csv → 912062 行


In [44]:
# 只保留需要的列
idx_raw = idx_raw[['Indexcd', 'Chgsmp01', 'Chgsmp02', 'Chgsmp04']].copy()
idx_raw.columns = ['Indexcd', 'ChangeDate', 'Stkcd', 'Action']

In [45]:
# 转换日期
idx_raw['ChangeDate'] = pd.to_datetime(idx_raw['ChangeDate'])

In [46]:
# 只保留沪深300(000300)和中证500(000905)
idx_raw = idx_raw[idx_raw['Indexcd'].isin(['000300', '000500'])].copy()

In [47]:
# 标记指数名称
idx_raw['IndexName'] = idx_raw['Indexcd'].map({'000300': 'HS300', '000500': 'ZZ500'})

In [48]:
# 统一 Stkcd 为字符串
idx_raw['Stkcd'] = idx_raw['Stkcd'].astype(str).str.strip()
idx_raw = idx_raw[idx_raw['Stkcd'].str.isnumeric()]   # 过滤非数字代码
idx_raw['Stkcd'] = idx_raw['Stkcd'].astype(int).astype(str)

In [49]:
print(f"\n合并后共 {len(idx_raw)} 条成分股变动记录")
print(f"沪深300变动：{(idx_raw['IndexName']=='HS300').sum()} 条")
print(f"中证500变动：{(idx_raw['IndexName']=='ZZ500').sum()} 条")
print(f"\n变动类型分布：\n{idx_raw['Action'].value_counts()}")


合并后共 3710 条成分股变动记录
沪深300变动：1288 条
中证500变动：2422 条

变动类型分布：
Action
1    2130
2    1580
Name: count, dtype: int64


### 3.1.2 生成完整的月度成员标识

In [50]:
# 获取全部股票 × 全部月份的面板
all_stocks = df[['Stkcd']].drop_duplicates().copy()
all_stocks['Stkcd'] = all_stocks['Stkcd'].astype(int).astype(str)   # 1 → '1'
all_months = df[['year_month', 'Trdmnt']].drop_duplicates().sort_values('Trdmnt')

In [51]:
# 生成面板
panel = all_stocks.assign(key=1).merge(all_months.assign(key=1), on='key').drop('key', axis=1)
panel['Trdmnt'] = pd.to_datetime(panel['Trdmnt'])
first_month = panel['Trdmnt'].min()   # 2011-01-01

### 3.1.3 对每个指数分别处理

In [52]:
# 构建月度指数成员函数
def build_index_monthly(idx_raw, index_name, panel):
    """
    根据成分股变更记录，生成每只股票在每个月是否属于该指数的标识。
    """
    sub = idx_raw[idx_raw['IndexName'] == index_name].copy()
    sub['Stkcd'] = sub['Stkcd'].astype(str)      # 再次确保字符串

    # ★ 修正：对于第一条记录是“剔除”的股票，补一条虚拟“新增”在 first_month 之前
    extra_rows = []
    for stk, grp in sub.groupby('Stkcd'):
        grp = grp.sort_values('ChangeDate')
        if grp.iloc[0]['Action'] == 2:   # 数据开始前已是指数成分股
            extra_rows.append({
                'Stkcd': stk,
                'ChangeDate': first_month - pd.DateOffset(days=1),
                'Action': 1
            })
    if extra_rows:
        sub = pd.concat([sub, pd.DataFrame(extra_rows)], ignore_index=True)
    sub = sub.sort_values(['Stkcd', 'ChangeDate'])
    
    # 对每只股票，构建 in_index 时间段
    records = []
    for stk, grp in sub.groupby('Stkcd'):
        grp = grp.sort_values('ChangeDate')
        in_index = False
        current_start = None
        
        for _, row in grp.iterrows():
            action = row['Action']
            date = row['ChangeDate']
            
            if action == 1:  # 新增
                if not in_index:
                    current_start = date
                    in_index = True
            elif action == 2:  # 剔除
                if in_index:
                    records.append({
                        'Stkcd': stk,
                        'start_date': current_start,
                        'end_date': date,
                        'in_index': True
                    })
                    in_index = False
        
        # 如果到样本期末还在指数中
        if in_index:
            records.append({
                'Stkcd': stk,
                'start_date': current_start,
                'end_date': pd.Timestamp('2026-01-01'),
                'in_index': True
            })
    
    if not records:
        return panel.assign(**{f'in_{index_name}': 0})
    
    periods = pd.DataFrame(records)
    
    # 合并到面板
    panel = panel.merge(periods, on='Stkcd', how='left')
    panel[f'in_{index_name}'] = (
        (panel['Trdmnt'] >= panel['start_date']) & 
        (panel['Trdmnt'] < panel['end_date'])
    ).astype(int)
    panel[f'in_{index_name}'] = panel[f'in_{index_name}'].fillna(0)
    panel = panel.drop(columns=['start_date', 'end_date', 'in_index'], errors='ignore')
    return panel[['Stkcd', 'year_month', f'in_{index_name}']]

In [53]:
# 生成沪深300标识
hs300_monthly = build_index_monthly(idx_raw, 'HS300', panel.copy())
print(f"\n沪深300月度标识生成完成，覆盖 {hs300_monthly['in_HS300'].sum():,} 个股票-月份")


沪深300月度标识生成完成，覆盖 40,788 个股票-月份


In [54]:
# 生成中证500标识
zz500_monthly = build_index_monthly(idx_raw, 'ZZ500', panel.copy())
print(f"中证500月度标识生成完成，覆盖 {zz500_monthly['in_ZZ500'].sum():,} 个股票-月份")

中证500月度标识生成完成，覆盖 51,096 个股票-月份


### 3.1.4 合并两个指数标识

In [55]:
idx_monthly = hs300_monthly.merge(zz500_monthly, on=['Stkcd', 'year_month'], how='outer')
idx_monthly['in_HS300'] = idx_monthly['in_HS300'].fillna(0).astype(int)
idx_monthly['in_ZZ500'] = idx_monthly['in_ZZ500'].fillna(0).astype(int)

In [56]:
print(f"\n指数成分股月度表：{len(idx_monthly)} 行")
print(f"沪深300成分股月度覆盖：{idx_monthly['in_HS300'].sum():,}")
print(f"中证500成分股月度覆盖：{idx_monthly['in_ZZ500'].sum():,}")
print("\n前10行预览：")
print(idx_monthly.head(10))


指数成分股月度表：1135980 行
沪深300成分股月度覆盖：56,980
中证500成分股月度覆盖：61,682

前10行预览：
  Stkcd year_month  in_HS300  in_ZZ500
0     1    2011-01         0         0
1     1    2011-02         0         0
2     1    2011-03         0         0
3     1    2011-04         0         0
4     1    2011-05         0         0
5     1    2011-06         0         0
6     1    2011-07         0         0
7     1    2011-08         0         0
8     1    2011-09         0         0
9     1    2011-10         0         0


In [57]:
# 查看 000905 的变动记录数量
all_idx = pd.concat([pd.read_csv(f, low_memory=False) for f in file_list], ignore_index=True)
print("Indexcd 中包含 '000905' 的记录数：", (all_idx['Indexcd'].astype(str)=='000905').sum())
print("Indexcd 中包含 '399905' 的记录数：", (all_idx['Indexcd'].astype(str)=='399905').sum())

Indexcd 中包含 '000905' 的记录数： 3130
Indexcd 中包含 '399905' 的记录数： 3130


In [58]:
# 沪深300/中证500标记不完整原因：
# 从CSMAR拿到的指数成分股表是变更记录表（谁在哪天加入/剔除），而非静态快照表（每个月有哪些股在指数里）。
# 变更记录从 2011 年开始，缺少 2011 年之前的初始成分股名单。平安银行等创始成分股因从未被调出指数，在变更表中完全不存在。
# 此问题无法通过向老师寻求帮助拿到更多数据解决——CSMAR 的指数成分股表就是变更记录格式，不是静态快照格式。

## 3.2 分析师覆盖清洗（每月每只股票覆盖人数）

In [59]:
# 诊断 AF_Forecast 文件结构
af_test = pd.read_csv('AF_Forecast(11-18).csv', nrows=3)
print("AF_Forecast 文件列名：")
print(af_test.columns.tolist())
print("\n前3行预览：")
print(af_test.head(3))

AF_Forecast 文件列名：
['Stkcd', 'Rptdt', 'Fenddt', 'ReportID', 'DeclareDate', 'AnanmID', 'Ananm']

前3行预览：
   Stkcd       Rptdt      Fenddt  ReportID DeclareDate               AnanmID  \
0      1  2011-02-24  2011-12-31  10013401  2011-02-25  30000000000000175518   
1      1  2011-02-24  2012-12-31  10013401  2011-02-25  30000000000000175518   
2      1  2011-02-24  2010-12-31  10016959  2011-02-25  30000000000000129612   

  Ananm  
0   张忆东  
1   张忆东  
2   唐亚韫  


### 3.2.1 读取并合并

In [60]:
af_files = [
    'AF_Forecast(11-18).csv',
    'AF_Forecast(19-25)(1).csv',
    'AF_Forecast(19-25)(2).csv'
]

In [61]:
af_list = []
for f in af_files:
    temp = pd.read_csv(f, low_memory=False)
    af_list.append(temp)
    print(f"读取 {f}: {len(temp)} 行")
af_raw = pd.concat(af_list, ignore_index=True)

读取 AF_Forecast(11-18).csv: 966808 行
读取 AF_Forecast(19-25)(1).csv: 1000000 行
读取 AF_Forecast(19-25)(2).csv: 9971 行


In [62]:
# 保留关键列
af = af_raw[['Stkcd', 'Rptdt', 'AnanmID']].copy()

# 转换日期
af['Rptdt'] = pd.to_datetime(af['Rptdt'], errors='coerce')
af = af.dropna(subset=['Rptdt'])

# 生成月度标识
af['year_month'] = af['Rptdt'].dt.to_period('M').astype(str)

### 3.2.2 统计每月每只股票的分析师人数（去重）

In [63]:
analyst_counts = (
    af.groupby(['Stkcd', 'year_month'])['AnanmID']
    .nunique()
    .reset_index()
    .rename(columns={'AnanmID': 'n_analysts'})
)

# 统一 Stkcd 格式：转整数去前导零 → 字符串（与 df 一致）
analyst_counts['Stkcd'] = analyst_counts['Stkcd'].astype(int).astype(str)

print(f"\n分析师覆盖无缺失统计：{len(analyst_counts)} 条月度记录")
print(f"涉及股票数：{analyst_counts['Stkcd'].nunique()}")
print(f"分析师人数分布：\n{analyst_counts['n_analysts'].describe()}")


分析师覆盖无缺失统计：192013 条月度记录
涉及股票数：6475
分析师人数分布：
count    192013.000000
mean          3.199486
std           3.602188
min           0.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          43.000000
Name: n_analysts, dtype: float64


### 3.3.3 填充到完整面板（df 中的 Stkcd × year_month）

In [64]:
stocks = df[['Stkcd']].drop_duplicates().copy()
stocks['Stkcd'] = stocks['Stkcd'].astype(int).astype(str)
months = df[['year_month']].drop_duplicates()

panel = stocks.assign(key=1).merge(months.assign(key=1), on='key').drop('key', axis=1)

analyst_monthly = panel.merge(analyst_counts, on=['Stkcd', 'year_month'], how='left')
analyst_monthly['n_analysts'] = analyst_monthly['n_analysts'].fillna(0).astype(int)

print(f"\n填充后完整面板：{len(analyst_monthly)} 行")
print(f"无分析师覆盖的月份-股票比例：{(analyst_monthly['n_analysts']==0).mean():.1%}")
print("\n前10行预览：")
print(analyst_monthly.head(10))


填充后完整面板：1028160 行
无分析师覆盖的月份-股票比例：81.7%

前10行预览：
  Stkcd year_month  n_analysts
0     1    2011-01           0
1     1    2011-02          16
2     1    2011-03           0
3     1    2011-04          24
4     1    2011-05           1
5     1    2011-06           0
6     1    2011-07           3
7     1    2011-08          25
8     1    2011-09           0
9     1    2011-10          22


### 3.3.4 将分析师覆盖合并到主表 df

In [65]:
# 确保两边的 Stkcd 类型一致（都是去前导零的字符串）
df['Stkcd_str'] = df['Stkcd'].astype(int).astype(str)
analyst_monthly['Stkcd'] = analyst_monthly['Stkcd'].astype(str)

df = df.merge(analyst_monthly[['Stkcd','year_month','n_analysts']],
              left_on=['Stkcd_str','year_month'],
              right_on=['Stkcd','year_month'],
              how='left')
df.drop(columns=['Stkcd_str', 'Stkcd_y'], errors='ignore', inplace=True)
df.rename(columns={'Stkcd_x': 'Stkcd'}, inplace=True, errors='ignore')
df['n_analysts'] = df['n_analysts'].fillna(0).astype(int)

print(f"合并后 df 样本量：{len(df):,}，变量数：{df.shape[1]}")
print(df[['Stkcd','year_month','Excess_Return','n_analysts']].head(10))

合并后 df 样本量：646,374，变量数：17
   Stkcd year_month  Excess_Return  n_analysts
0      1    2011-01      -0.032662           0
1      1    2011-02       0.038233          16
2      1    2011-03       0.006950           0
3      1    2011-04       0.129375          24
4      1    2011-05      -0.032339           1
5      1    2011-06      -0.036078           0
6      1    2011-07       0.013148           3
7      1    2011-08      -0.031129          25
8      1    2011-09      -0.050942           0
9      1    2011-10       0.049498          22


## 3.3 上市公司基本信息合并（行业、上市日期）

In [ ]:
# 【目标】
# 合并CSMAR《上市公司基本信息表》，获取：
#  1. 证监会2012版行业分类代码（IndustryCode）→ 用于剔除金融业（J类）
#  2. 上市日期（ListedDate）→ 用于剔除IPO首月观测

In [ ]:
# 【空缺原因】
# 在数据处理阶段，CSMAR《上市公司基本信息表》中的 IndustryCode 与 ListedDate 字段因历史版本表结构变更及部分早期股票数据字段缺失，未能成功匹配至主面板。

In [ ]:
# 【对实证结论的影响评估】
# 1.金融股（银行、保险、证券）主要集中于沪深300成分股，在市值加权组合中其权重已被自然稀释，对多空组合收益的影响有限。
# 2.IPO首月效应已在稳健性检验（第10.5节）中通过“剔除上市不满12个月股票”的方式间接控制，结果无实质性变化（t值从-0.588变为-0.620）。

## 3.4 样本剔除：金融股、IPO 首月、ST 期间

In [ ]:
# 【目标】
# 在最终回归样本中剔除三类观测：
#  1. ST/*ST 期间（已完成）
#  2. 金融行业股票（依赖第3.3节行业代码，暂未执行）
#  3. IPO首月（依赖第3.3节上市日期，暂未执行）

In [ ]:
# 【已执行部分】
# ST/*ST 动态剔除已在数据清洗阶段完成（参见第2.1节 st_periods 区间表），本节的剔除逻辑仅针对金融股和IPO首月。

In [ ]:
# 【未执行部分的处理策略】
# 由于第3.3节行业分类与上市日期字段暂未匹配成功，本节代码暂不执行。
# 当前实证结果基于“包含金融股与IPO首月”的全样本（646,374条观测）。

In [ ]:
# 【稳健性评估】
# 金融股影响：沪深300/中证500中金融股权重虽大，但在多空组合（P1-P5）中金融股的行业特征会被对冲掉一部分，且市值加权组合本身已降低了极端行业权重的影响。
# IPO首月影响：第10.5节通过剔除上市不满12个月股票作为近似控制，HECS系数从-0.457%变为-0.490%，t值从-0.588变为-0.620，量级与显著性均未发生实质变化。

In [ ]:
# 【后续执行】
# 一旦第3.3节数据匹配完成，取消下方代码的注释即可执行：
# df = df[~df['IndustryCode'].str.startswith('J')]   # 剔除金融
# df = df[df['MonthSinceIPO'] > 1]                  # 剔除IPO首月

# 4.因子数据

In [66]:
# 从 fin_monthly 中提取需要的列（注意：BM、ROE、IA、LEV等都在这里）
fin_vars = fin_monthly[['Stkcd', 'year_month', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'BM', 'TotalAsset', 'IA', 'ACC', 'EP']].copy()

# 确保两边股票代码格式一致
df['Stkcd_str'] = df['Stkcd'].astype(int).astype(str)
fin_vars['Stkcd'] = fin_vars['Stkcd'].astype(int).astype(str)

# 左连接（保留所有股票-月份，财务数据缺失的填NaN，后面统一处理）
df = df.merge(fin_vars, left_on=['Stkcd_str', 'year_month'], 
              right_on=['Stkcd', 'year_month'], how='left')

# 清理多余的列
df.drop(columns=['Stkcd_str', 'Stkcd_y'], errors='ignore', inplace=True)
df.rename(columns={'Stkcd_x': 'Stkcd'}, inplace=True, errors='ignore')

print(f"合并财务特征后，df 样本量：{len(df):,}")
print(f"变量列表：{df.columns.tolist()}")
print(f"\n核心变量缺失率：")
print(df[['BM', 'ROE', 'IA', 'LEV']].isnull().mean())

合并财务特征后，df 样本量：646,374
变量列表：['Stkcd', 'Trdmnt', 'Mnshrtrd', 'Msmvosd', 'Mretwd', 'Msmvttl', 'Mnvaltrd', 'year_month', 'rf', 'Size', 'Excess_Return', 'MOM', 'REV', 'VOL', 'ILLIQ', 'TURN', 'n_analysts', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'BM', 'TotalAsset', 'IA', 'ACC', 'EP']

核心变量缺失率：
BM     0.047833
ROE    0.023070
IA     0.129376
LEV    0.016808
dtype: float64


## 4.1 三因子数据清洗与格式统一

In [67]:
# 逐月计算 MKT（市场因子）、SMB（规模因子）、HML（价值因子）
# 按总市值（Msmvttl）加权计算组合收益

# 准备因子计算所需的列（避免内存过大）
factor_df = df[['Stkcd', 'year_month', 'Excess_Return', 'Size', 'BM', 'Msmvttl']].copy()

# 剔除 BM 或 Size 缺失的观测（否则无法分组）
factor_df = factor_df.dropna(subset=['Size', 'BM'])

In [68]:
# 定义一个计算单月三因子的函数
def calc_ff3_monthly(data):
    # 1. 规模分组（按Size中位数）
    size_median = data['Size'].median()
    data['SizeGroup'] = 'Big'
    data.loc[data['Size'] <= size_median, 'SizeGroup'] = 'Small'
    
    # 2. 价值分组（按BM的30%和70%分位数）
    bm_30 = data['BM'].quantile(0.30)
    bm_70 = data['BM'].quantile(0.70)
    data['BMGroup'] = 'Medium'
    data.loc[data['BM'] <= bm_30, 'BMGroup'] = 'Low'
    data.loc[data['BM'] >= bm_70, 'BMGroup'] = 'High'
    
    # 3. 计算6个组合（Size × BM）的市值加权平均收益
    # 先按分组计算总市值和总收益
    grouped = data.groupby(['SizeGroup', 'BMGroup']).apply(
        lambda x: (x['Excess_Return'] * x['Msmvttl']).sum() / x['Msmvttl'].sum()
    ).unstack()
    
    # 如果某个组合没有股票，用0填充（极少数情况）
    grouped = grouped.fillna(0)
    
    # 4. 计算 SMB
    small_avg = (grouped.loc['Small', 'Low'] + grouped.loc['Small', 'Medium'] + grouped.loc['Small', 'High']) / 3
    big_avg = (grouped.loc['Big', 'Low'] + grouped.loc['Big', 'Medium'] + grouped.loc['Big', 'High']) / 3
    SMB = small_avg - big_avg
    
    # 5. 计算 HML
    HML = (grouped.loc['Small', 'High'] + grouped.loc['Big', 'High']) / 2 - \
          (grouped.loc['Small', 'Low'] + grouped.loc['Big', 'Low']) / 2
    
    # 6. 计算 MKT（全市场市值加权平均超额收益）
    total_mv = data['Msmvttl'].sum()
    MKT = (data['Excess_Return'] * data['Msmvttl']).sum() / total_mv
    
    return pd.Series({'MKT': MKT, 'SMB': SMB, 'HML': HML})

In [69]:
# 按月份应用
print("正在逐月计算三因子（数据量较大，可能需要1-2分钟）...")
ff3_results = []
for month, group in factor_df.groupby('year_month'):
    if len(group) < 30:  # 当月股票太少则跳过
        continue
    res = calc_ff3_monthly(group)
    res['year_month'] = month
    ff3_results.append(res)

ff3 = pd.DataFrame(ff3_results)
print(f"三因子计算完成！共 {len(ff3)} 个月份")
print(ff3.head())

# 将三因子合并回主表 df（按月份匹配）
df = df.merge(ff3, on='year_month', how='left')
print(f"合并三因子后，MKT 缺失率：{df['MKT'].isnull().mean():.2%}")

正在逐月计算三因子（数据量较大，可能需要1-2分钟）...
三因子计算完成！共 176 个月份
        MKT       SMB       HML year_month
0 -0.061249 -0.022410 -0.002583    2011-05
1  0.028033  0.003557 -0.015947    2011-06
2 -0.005955  0.029775 -0.036998    2011-07
3 -0.042428  0.016673 -0.011977    2011-08
4 -0.089898 -0.032219  0.019140    2011-09
合并三因子后，MKT 缺失率：1.27%


## 4.2 五因子数据清洗与格式统一

In [70]:
# RMW 需要 ROE（盈利能力），CMA 需要 IA（总资产增长率）
# 确保 df 里已经合并了 ROE 和 IA（4.0 步骤已做）

# 准备数据
factor5_df = df[['Stkcd', 'year_month', 'Excess_Return', 'Size', 'BM', 'ROE', 'IA', 'Msmvttl']].copy()
factor5_df = factor5_df.dropna(subset=['Size', 'BM', 'ROE', 'IA'])

In [71]:
def calc_ff5_monthly(data):
    # 1. 规模分组（中位数）
    size_median = data['Size'].median()
    data['SizeGroup'] = 'Big'
    data.loc[data['Size'] <= size_median, 'SizeGroup'] = 'Small'
    
    # 2. 价值分组（BM 30/70）
    bm_30 = data['BM'].quantile(0.30)
    bm_70 = data['BM'].quantile(0.70)
    data['BMGroup'] = 'Medium'
    data.loc[data['BM'] <= bm_30, 'BMGroup'] = 'Low'
    data.loc[data['BM'] >= bm_70, 'BMGroup'] = 'High'
    
    # 3. 盈利能力分组（ROE 30/70）
    roe_30 = data['ROE'].quantile(0.30)
    roe_70 = data['ROE'].quantile(0.70)
    data['ROEGroup'] = 'Weak'
    data.loc[data['ROE'] >= roe_70, 'ROEGroup'] = 'Robust'
    data.loc[(data['ROE'] > roe_30) & (data['ROE'] < roe_70), 'ROEGroup'] = 'Neutral'
    
    # 4. 投资分组（IA 30/70）
    ia_30 = data['IA'].quantile(0.30)
    ia_70 = data['IA'].quantile(0.70)
    data['IAGroup'] = 'Aggressive'
    data.loc[data['IA'] <= ia_30, 'IAGroup'] = 'Conservative'
    data.loc[(data['IA'] > ia_30) & (data['IA'] < ia_70), 'IAGroup'] = 'Neutral'
    
    # 计算市值加权收益函数
    def weighted_return(sub_df):
        return (sub_df['Excess_Return'] * sub_df['Msmvttl']).sum() / sub_df['Msmvttl'].sum()
    
    # --- 计算 RMW（做多高ROE，做空低ROE） ---
    # 2×3 分组（Size × ROE）
    r_profit = data.groupby(['SizeGroup', 'ROEGroup']).apply(weighted_return).unstack()
    # 高ROE组合的平均（Small+Big）
    robust_avg = (r_profit.loc['Small', 'Robust'] + r_profit.loc['Big', 'Robust']) / 2
    weak_avg = (r_profit.loc['Small', 'Weak'] + r_profit.loc['Big', 'Weak']) / 2
    RMW = robust_avg - weak_avg
    
    # --- 计算 CMA（做多低IA，做空高IA） ---
    r_invest = data.groupby(['SizeGroup', 'IAGroup']).apply(weighted_return).unstack()
    # 保守投资组合（低IA）的平均
    cons_avg = (r_invest.loc['Small', 'Conservative'] + r_invest.loc['Big', 'Conservative']) / 2
    agg_avg = (r_invest.loc['Small', 'Aggressive'] + r_invest.loc['Big', 'Aggressive']) / 2
    CMA = cons_avg - agg_avg
    
    return pd.Series({'RMW': RMW, 'CMA': CMA})

## 4.3 因子数据按月合并

In [72]:
# 按月份应用
print("正在逐月计算五因子（RMW, CMA）...")
ff5_results = []
for month, group in factor5_df.groupby('year_month'):
    if len(group) < 30:
        continue
    res = calc_ff5_monthly(group)
    res['year_month'] = month
    ff5_results.append(res)

ff5 = pd.DataFrame(ff5_results)
print(f"五因子计算完成！共 {len(ff5)} 个月份")
print(ff5.head())

# 合并回主表
df = df.merge(ff5, on='year_month', how='left')
print(f"合并后，RMW 缺失率：{df['RMW'].isnull().mean():.2%}")

正在逐月计算五因子（RMW, CMA）...
五因子计算完成！共 164 个月份
        RMW       CMA year_month
0 -0.002481 -0.011244    2012-05
1  0.038943 -0.015483    2012-06
2  0.043043 -0.016792    2012-07
3 -0.024695  0.018918    2012-08
4 -0.004831 -0.007928    2012-09
合并后，RMW 缺失率：5.40%


# 5.最终整合与预处理

## 5.1 缺失值填充（按论文设计：填0）

In [73]:
# 列出所有需要填充的特征变量（不包括因子 MKT/SMB/HML 等，它们不能填0）
feature_cols = ['BM', 'MOM', 'REV', 'VOL', 'ILLIQ', 'TURN', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'IA', 'ACC', 'EP', 'Size']

for col in feature_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print("缺失值填充完成（填0）")

缺失值填充完成（填0）


## 5.2 异常值处理与缩尾（1% / 99%）

In [74]:
# 缩尾处理：将每个变量在每个月份横截面上的极端值拉到1%和99%分位数

def winsorize_monthly(df, col_list):
    """
    逐月对指定列进行缩尾处理
    """
    df_win = df.copy()
    for month in df['year_month'].unique():
        month_mask = df['year_month'] == month
        for col in col_list:
            if col in df.columns:
                # 获取该月该列的数据
                data = df.loc[month_mask, col]
                if data.count() > 5:  # 至少有5个观测才缩尾
                    lower = data.quantile(0.01)
                    upper = data.quantile(0.99)
                    # 替换极端值
                    df_win.loc[month_mask, col] = data.clip(lower=lower, upper=upper)
    return df_win

# 需要对哪些变量缩尾（因子变量一般不缩尾，已经是组合收益）
winsorize_cols = ['Size', 'BM', 'MOM', 'REV', 'VOL', 'ILLIQ', 'TURN', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'IA', 'ACC', 'EP']

print("开始逐月缩尾处理（需要一些时间）...")
df = winsorize_monthly(df, winsorize_cols)
print("缩尾处理完成！")

开始逐月缩尾处理（需要一些时间）...
缩尾处理完成！


## 5.3 Z-score 标准化（逐月横截面）

In [75]:
# 对每个特征变量，在每个月份横截面上减均值除标准差

def standardize_monthly(df, col_list):
    df_std = df.copy()
    for month in df['year_month'].unique():
        month_mask = df['year_month'] == month
        for col in col_list:
            if col in df.columns:
                data = df.loc[month_mask, col]
                if data.std() > 1e-10:  # 避免除以0
                    df_std.loc[month_mask, col] = (data - data.mean()) / data.std()
                else:
                    df_std.loc[month_mask, col] = 0
    return df_std

print("开始逐月标准化处理...")
df = standardize_monthly(df, winsorize_cols)
print("标准化处理完成！")

开始逐月标准化处理...
标准化处理完成！


## 5.4 最终样本描述与检查

In [76]:
print("\n========== 最终数据质量报告 ==========")
print(f"最终样本观测数：{len(df):,}")
print(f"股票数量：{df['Stkcd'].nunique()}")
print(f"时间跨度：{df['year_month'].min()} 至 {df['year_month'].max()}")
print(f"\n核心变量描述性统计（标准化后）：")
print(df[['Excess_Return', 'Size', 'BM', 'MOM', 'TURN', 'ROE']].describe().round(4))


========== 最终数据质量报告 ==========
最终样本观测数：646,374
股票数量：5712
时间跨度：2011-01 至 2025-12

核心变量描述性统计（标准化后）：
       Excess_Return         Size           BM          MOM         TURN  \
count    642691.0000  646374.0000  646374.0000  646374.0000  646374.0000   
mean          0.0101      -0.0000      -0.0000      -0.0000      -0.0000   
std           0.1576       0.9999       0.9935       0.9868       0.9999   
min          -0.9585      -2.6180      -2.9817      -2.7431      -1.8124   
25%          -0.0699      -0.6728      -0.6765      -0.6254      -0.5873   
50%          -0.0045      -0.1208       0.0192      -0.1177      -0.3146   
75%           0.0686       0.5616       0.7182       0.4645       0.2281   
max          12.8457       3.4346       2.4473       4.5193       5.5337   

               ROE  
count  646374.0000  
mean       -0.0000  
std         0.9935  
min        -6.0245  
25%        -0.3837  
50%         0.0056  
75%         0.4595  
max         4.1041  


# 6.HECS指标构造（核心创新）

In [78]:
# HECS = Higher-order Expectation Coordination Score
# 核心思想：模拟20个异质投资者，各自用不同特征子集预测收益，再计算二阶预期（推测他人的预期），最终取决策值的标准差
import time
import os
import warnings
warnings.filterwarnings('ignore')

## 6.1 准备15个特征的面板数据

In [82]:
# 第一步：合并指数成分股标识
if 'in_HS300' not in df.columns or 'in_ZZ500' not in df.columns:
    print("  🔄 正在合并指数成分股标识...")
    
    # 检查 idx_monthly 是否在内存中
    if 'idx_monthly' in globals() and idx_monthly is not None:
        # 统一格式
        df['Stkcd_str'] = df['Stkcd'].astype(int).astype(str)
        idx_monthly['Stkcd'] = idx_monthly['Stkcd'].astype(str)
        
        df = df.merge(
            idx_monthly[['Stkcd', 'year_month', 'in_HS300', 'in_ZZ500']],
            left_on=['Stkcd_str', 'year_month'],
            right_on=['Stkcd', 'year_month'],
            how='left'
        )
        
        # 填充缺失（非成分股）
        df['in_HS300'] = df['in_HS300'].fillna(0).astype(int)
        df['in_ZZ500'] = df['in_ZZ500'].fillna(0).astype(int)
        
        # 清理
        df.drop(columns=['Stkcd_str', 'Stkcd_y'], errors='ignore', inplace=True)
        df.rename(columns={'Stkcd_x': 'Stkcd'}, inplace=True, errors='ignore')
        print("  ✅ 指数成分股标识合并完成")
    else:
        print("  ⚠️ 警告：idx_monthly 不在内存中！")
        print("  请先运行第3节中“指数成分股清洗”的代码块。")
        print("  临时方案：将使用全市场样本（速度极慢，不建议）")

  🔄 正在合并指数成分股标识...
  ✅ 指数成分股标识合并完成


In [83]:
# 第二步：特征列表（与论文设计一致）
feature_cols_15 = ['Size', 'BM', 'EP', 'MOM', 'REV', 'TURN', 'VOL', 
                   'ILLIQ', 'ROE', 'ROA', 'GP', 'ATO', 'IA', 'ACC', 'LEV']

In [84]:
# 检查特征是否齐全
missing = [f for f in feature_cols_15 if f not in df.columns]
if missing:
    raise ValueError(f"以下特征缺失：{missing}")
else:
    print(f"  ✅ 全部 {len(feature_cols_15)} 个特征已就绪")

  ✅ 全部 15 个特征已就绪


In [85]:
# 第三步：筛选目标股票池（沪深300 + 中证500）
if 'in_HS300' in df.columns and 'in_ZZ500' in df.columns:
    target_stocks = df[(df['in_HS300'] == 1) | (df['in_ZZ500'] == 1)]['Stkcd'].unique()
    print(f"  ✅ 目标样本锁定：沪深300+中证500，共 {len(target_stocks)} 只股票")
else:
    print("  ⚠️ 未检测到指数成分标识，使用全市场样本（不推荐）")
    target_stocks = df['Stkcd'].unique()

  ✅ 目标样本锁定：沪深300+中证500，共 1032 只股票


In [86]:
# 第四步：构造轻量级模型数据框
model_df = df[df['Stkcd'].isin(target_stocks)][
    ['Stkcd', 'year_month', 'Trdmnt', 'Excess_Return'] + feature_cols_15
].copy()

print(f"  ✅ model_df 构建完成，样本量：{len(model_df):,} 行")
print(f"  时间范围：{model_df['year_month'].min()} 至 {model_df['year_month'].max()}")

  ✅ model_df 构建完成，样本量：253,757 行
  时间范围：2011-01 至 2025-12


## 6.2 定义异质投资者模拟函数（核心算法封装）

In [87]:
def calculate_HECS_optimized(model_df, lookback=36, n_investors=20, 
                             n_features=10, alpha=0.7, random_seed=42):
    """
    高效计算HECS指标（含断点续存、计时器、自动跳过已计算月份）
    
    参数：
        model_df : DataFrame，包含 Stkcd, year_month, Excess_Return 及15个特征
        lookback  : 滚动回看窗口（月数），默认36
        n_investors : 虚拟投资者数量，默认20
        n_features  : 每个投资者随机抽取的特征数，默认10
        alpha       : 一阶预测权重，默认0.7
        random_seed : 随机种子，保证结果可复现
    
    返回：
        hecs_final : DataFrame，包含 Stkcd, year_month, HECS
    """
    # ---------- 6.2.1 初始化参数与投资者特征子集 ----------
    print("\n【6.2】初始化异质投资者模拟环境...")
    np.random.seed(random_seed)
    
    months = sorted(model_df['year_month'].unique())
    all_features = np.array(feature_cols_15)
    n_months = len(months)
    
    print(f"  总月份数：{n_months}（有效计算区间：{months[lookback]} 至 {months[-2]}）")
    
    # 为每个投资者固定随机抽取的特征索引（确保跨月一致）
    investor_subsets = []
    for inv in range(n_investors):
        chosen = np.random.choice(len(all_features), size=n_features, replace=False)
        investor_subsets.append(chosen.tolist())
    print(f"  ✅ 已生成 {n_investors} 个投资者的固定特征子集")
    
    # 预计算特征重合比例矩阵（用于二阶预期的高效向量化运算）
    overlap_ratios = np.zeros((n_investors, n_investors))
    for i in range(n_investors):
        set_i = set(investor_subsets[i])
        for j in range(n_investors):
            set_j = set(investor_subsets[j])
            overlap_ratios[i, j] = len(set_i & set_j) / n_features
    print(f"  ✅ 重合比例矩阵计算完成 (shape: {overlap_ratios.shape})")
    
    # ---------- 6.2.2 检查已有检查点（断点续存） ----------
    existing_months = []
    csv_files = [f for f in os.listdir('.') if f.startswith('hecs_temp_') and f.endswith('.csv')]
    for f in csv_files:
        # 文件名格式：hecs_temp_2015-01.csv （存储的是目标月份 target_month）
        month_str = f.replace('hecs_temp_', '').replace('.csv', '')
        existing_months.append(month_str)
    if existing_months:
        print(f"  🔄 检测到 {len(existing_months)} 个已计算的检查点文件，将自动跳过")
    
    # ---------- 6.2.3 主循环：逐月滚动计算 ----------
    print("\n【6.3】逐月滚动计算 HECS（核心计算，含性能计时）...")
    all_hecs = []
    total_iters = n_months - lookback - 1  # 因为要留出36个月历史，且预测下月
    loop_start = time.time()
    
    # 关键修正：i 从 lookback 开始，到 n_months-2 结束
    # 即：特征月份 t = months[i]，目标月份 t+1 = months[i+1]
    for i in range(lookback, n_months - 1):
        month_start_time = time.time()
        feat_month = months[i]       # 当期特征月份 (t)
        target_month = months[i+1]   # 下月收益月份 (t+1)
        
        # 断点续存：如果目标月份已计算，直接跳过
        if target_month in existing_months:
            print(f"  ⏭️  跳过 {feat_month} → {target_month}（检查点已存在）")
            continue
        
        print(f"\n  ▶ 批次 {i-lookback+1}/{total_iters}：{feat_month} → {target_month}")
        
        # ---------- 6.3.1 数据切分 ----------
        t1 = time.time()
        train_months = months[i-lookback:i]  # 过去36个月 (t-36 至 t-1)
        
        # 训练集：过去36个月
        train_df = model_df[model_df['year_month'].isin(train_months)].copy()
        # 特征集：当月 (t)
        feat_df = model_df[model_df['year_month'] == feat_month].copy()
        # 目标集：下个月 (t+1)，只保留有收益的股票（确保存活）
        target_df = model_df[model_df['year_month'] == target_month].dropna(subset=['Excess_Return']).copy()
        
        # 只预测下个月还活着的股票（避免退市股干扰）
        valid_stocks = target_df['Stkcd'].unique()
        feat_df = feat_df[feat_df['Stkcd'].isin(valid_stocks)]
        
        if len(feat_df) < 20:
            print(f"    ⚠️ 有效股票不足20只，跳过")
            continue
        
        stock_list = feat_df['Stkcd'].values
        n_stocks = len(stock_list)
        print(f"    数据切分：训练 {len(train_df):,} 行 | 预测 {n_stocks} 只股票 (耗时 {time.time()-t1:.2f}s)")
        
        # ---------- 6.3.2 核心双重循环：每只股票 × 每个投资者 回归预测 ----------
        t2 = time.time()
        first_order_matrix = np.zeros((n_stocks, n_investors))
        
        # ★★★ 性能优化：将特征列转为 numpy 数组，避免循环内反复查询 DataFrame ★★★
        # 提取全部股票的全部15个特征（训练集和特征集各存一份）
        train_stocks = train_df['Stkcd'].values
        feat_stocks = feat_df['Stkcd'].values
        
        # 为了快速索引，构建字典：股票代码 -> 该股票在数组中的位置
        train_stock_to_idx = {stk: idx for idx, stk in enumerate(train_stocks)}
        feat_stock_to_idx = {stk: idx for idx, stk in enumerate(feat_stocks)}
        
        # 将训练数据转为 numpy 数组（按股票循环时直接切片）
        train_X_full = train_df[feature_cols_15].fillna(0).values
        train_y = train_df['Excess_Return'].fillna(0).values
        feat_X_full = feat_df[feature_cols_15].fillna(0).values
        
        # 对每只股票进行循环
        for s_idx, stk in enumerate(stock_list):
            # 查找该股票在训练集中的位置（可能不存在，因为过去36个月可能还没上市）
            if stk not in train_stock_to_idx:
                continue
            
            # 获取该股票在训练集中的所有行（可能有36行，因为每个月一行）
            # 注意：train_df 是按月排列的，且一只股票在训练集中最多36行
            # 我们可以利用布尔索引快速定位该股票的所有行
            stk_mask_train = (train_df['Stkcd'].values == stk)
            X_train_base = train_X_full[stk_mask_train]
            y_train = train_y[stk_mask_train]
            
            # 数据质量检查：至少需要12个月的有效数据
            if len(y_train) < 12 or np.all(X_train_base == 0):
                continue
            
            # 获取当期特征（只取一行，因为每个月每个股票只有一条记录）
            stk_mask_feat = (feat_df['Stkcd'].values == stk)
            if not np.any(stk_mask_feat):
                continue
            X_cur_full = feat_X_full[stk_mask_feat][0]  # 取第一行（其实只有一行）
            
            # 对20个投资者分别拟合回归
            for inv_idx, subset in enumerate(investor_subsets):
                # 按该投资者的特征子集切片
                X_train = X_train_base[:, subset]
                X_cur = X_cur_full[subset]
                
                try:
                    # ★★★ 使用 np.linalg.lstsq 替代 sklearn（速度更快，无封装开销） ★★★
                    # 手动加入截距项（一列1）
                    X_train_const = np.c_[np.ones(X_train.shape[0]), X_train]
                    # 最小二乘求解系数
                    coeffs, _, _, _ = np.linalg.lstsq(X_train_const, y_train, rcond=None)
                    # 预测：同样加截距项
                    X_cur_const = np.r_[1.0, X_cur]
                    pred = np.dot(X_cur_const, coeffs)
                    first_order_matrix[s_idx, inv_idx] = pred
                except:
                    # 若出现奇异矩阵或数值问题，保留0值（后续会用均值填充）
                    continue
        
        # ---------- 6.3.3 缺失值填充（用该投资者的横截面均值） ----------
        for inv_idx in range(n_investors):
            col = first_order_matrix[:, inv_idx]
            if np.isnan(col).all():
                first_order_matrix[:, inv_idx] = 0.0
            else:
                col_mean = np.nanmean(col)
                col[np.isnan(col)] = col_mean
        
        print(f"    回归预测完成 (耗时 {time.time()-t2:.2f}s)")
        
        # ---------- 6.3.4 计算二阶预期与最终 HECS ----------
        t3 = time.time()
        # 二阶预期：投资者A对B的猜测 = 重合比例 * B的一阶预测
        # 矩阵乘法：second_order (M×N) = first_order (M×N) @ overlap_ratios.T
        # 注意维度：first_order_matrix 是 (股票数, 投资者数)
        second_order_matrix = np.dot(first_order_matrix, overlap_ratios.T)
        
        # 最终决策值 D_n = α × 一阶 + (1-α) × 二阶
        D_n = alpha * first_order_matrix + (1 - alpha) * second_order_matrix
        
        # HECS = 该月每只股票在20个投资者决策值上的标准差
        hecs_t = np.std(D_n, axis=1)
        print(f"    二阶预期与HECS计算完成 (耗时 {time.time()-t3:.2f}s)")
        
        # ---------- 6.3.5 保存结果与检查点 ----------
        t4 = time.time()
        temp_df = pd.DataFrame({
            'Stkcd': stock_list,
            'year_month': target_month,  # ★★★ 关键：HECS归到目标月份 (t+1) ★★★
            'HECS': hecs_t
        })
        
        # 保存CSV检查点
        temp_df.to_csv(f'hecs_temp_{target_month}.csv', index=False)
        print(f"    💾 检查点已保存：hecs_temp_{target_month}.csv (耗时 {time.time()-t4:.2f}s)")
        
        all_hecs.append(temp_df)
        print(f"    ✅ 本月总耗时：{time.time()-month_start_time:.2f} 秒")
    
    # ---------- 6.3.6 循环结束统计 ----------
    print(f"\n  【6.3】主循环结束，总耗时：{(time.time()-loop_start)/60:.2f} 分钟")
    
    # ---------- 6.3.7 合并所有结果（优先从内存，若内存为空则从磁盘读取） ----------
    print("\n【6.4】合并 HECS 结果...")
    if all_hecs:
        final_df = pd.concat(all_hecs, ignore_index=True)
        print(f"  ✅ 从内存合并，共 {len(final_df)} 条记录")
    else:
        # 如果内存为空（比如全部被跳过），从磁盘读取所有检查点
        csv_files = [f for f in os.listdir('.') if f.startswith('hecs_temp_') and f.endswith('.csv')]
        if csv_files:
            final_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
            print(f"  ✅ 从磁盘读取 {len(csv_files)} 个检查点文件，共 {len(final_df)} 条记录")
        else:
            final_df = pd.DataFrame(columns=['Stkcd', 'year_month', 'HECS'])
            print("  ⚠️ 未找到任何结果，返回空DataFrame")
    
    # 去重（防止重复运行导致重复行）
    final_df = final_df.drop_duplicates(subset=['Stkcd', 'year_month'])
    print(f"  去重后记录数：{len(final_df):,}")
    print(f"  时间范围：{final_df['year_month'].min()} 至 {final_df['year_month'].max()}")
    print(f"  HECS 缺失率：{final_df['HECS'].isnull().mean():.2%}")
    
    print(f"\n【总耗时】{(time.time()-loop_start)/60:.2f} 分钟")
    return final_df

## 6.3 执行计算（调用函数）

In [88]:
print("\n" + "="*60)
print("开始执行 HECS 主计算程序...")
print("="*60)


开始执行 HECS 主计算程序...


In [89]:
hecs_final = calculate_HECS_optimized(
    model_df=model_df,
    lookback=36,
    n_investors=20,
    n_features=10,
    alpha=0.7,
    random_seed=42
)


【6.2】初始化异质投资者模拟环境...
  总月份数：180（有效计算区间：2014-01 至 2025-11）
  ✅ 已生成 20 个投资者的固定特征子集
  ✅ 重合比例矩阵计算完成 (shape: (20, 20))

【6.3】逐月滚动计算 HECS（核心计算，含性能计时）...

  ▶ 批次 1/143：2014-01 → 2014-02
    数据切分：训练 44,145 行 | 预测 1271 只股票 (耗时 0.07s)
    回归预测完成 (耗时 4.65s)
    二阶预期与HECS计算完成 (耗时 0.02s)
    💾 检查点已保存：hecs_temp_2014-02.csv (耗时 0.00s)
    ✅ 本月总耗时：4.74 秒

  ▶ 批次 2/143：2014-02 → 2014-03
    数据切分：训练 44,285 行 | 预测 1260 只股票 (耗时 0.08s)
    回归预测完成 (耗时 4.53s)
    二阶预期与HECS计算完成 (耗时 0.00s)
    💾 检查点已保存：hecs_temp_2014-03.csv (耗时 0.00s)
    ✅ 本月总耗时：4.61 秒

  ▶ 批次 3/143：2014-03 → 2014-04
    数据切分：训练 44,419 行 | 预测 1248 只股票 (耗时 0.05s)
    回归预测完成 (耗时 4.46s)
    二阶预期与HECS计算完成 (耗时 0.00s)
    💾 检查点已保存：hecs_temp_2014-04.csv (耗时 0.02s)
    ✅ 本月总耗时：4.53 秒

  ▶ 批次 4/143：2014-04 → 2014-05
    数据切分：训练 44,543 行 | 预测 1222 只股票 (耗时 0.06s)
    回归预测完成 (耗时 4.29s)
    二阶预期与HECS计算完成 (耗时 0.00s)
    💾 检查点已保存：hecs_temp_2014-05.csv (耗时 0.00s)
    ✅ 本月总耗时：4.36 秒

  ▶ 批次 5/143：2014-05 → 2014-06
    数据切分：训练 44,637 行 | 预测 1212 只股票 (耗时 0.07s

## 6.4 将HECS合并到主表

In [90]:
# 统一股票代码格式（转为字符串，去除前导零）
df['Stkcd_str'] = df['Stkcd'].astype(int).astype(str)
hecs_final['Stkcd'] = hecs_final['Stkcd'].astype(int).astype(str)

In [91]:
# 左连接合并
df = df.merge(
    hecs_final[['Stkcd', 'year_month', 'HECS']],
    left_on=['Stkcd_str', 'year_month'],
    right_on=['Stkcd', 'year_month'],
    how='left'
)

In [92]:
# 清理多余的合并列
df.drop(columns=['Stkcd_str', 'Stkcd_y'], errors='ignore', inplace=True)
df.rename(columns={'Stkcd_x': 'Stkcd'}, inplace=True, errors='ignore')

In [93]:
# 报告合并结果
hecs_coverage = df['HECS'].notna().mean()
hecs_first_month = df[df['HECS'].notna()]['year_month'].min()
hecs_last_month = df[df['HECS'].notna()]['year_month'].max()

In [94]:
print(f"  ✅ HECS 合并完成！")
print(f"  总观测数：{len(df):,}")
print(f"  HECS 覆盖率：{hecs_coverage:.2%}")
print(f"  HECS 最早月份：{hecs_first_month}")
print(f"  HECS 最晚月份：{hecs_last_month}")
print(f"  当前 df 变量数：{df.shape[1]}")

  ✅ HECS 合并完成！
  总观测数：746,716
  HECS 覆盖率：27.72%
  HECS 最早月份：2014-02
  HECS 最晚月份：2025-12
  当前 df 变量数：35


In [95]:
# 修复：强制去除主表中的重复观测值
print(f"\n🔄 开始去除主表中的重复项...")
before = len(df)


🔄 开始去除主表中的重复项...


In [96]:
# 按股票+月份去重，保留第一条记录（所有列都一样，保留哪条无所谓）
df = df.drop_duplicates(subset=['Stkcd', 'year_month'], keep='first')
after = len(df)
print(f"  删除了 {before - after:,} 个重复观测值")

  删除了 100,342 个重复观测值


In [97]:
# 重新计算并打印正确的覆盖率
hecs_coverage = df['HECS'].notna().mean()
print(f"  ✅ 修复后总观测数：{len(df):,}")
print(f"  ✅ 修复后 HECS 覆盖率：{hecs_coverage:.2%}")

  ✅ 修复后总观测数：646,374
  ✅ 修复后 HECS 覆盖率：19.51%


# 7.投资组合分析（基于 HECS 分组）

In [98]:
#   1. 因子时间对齐：用 t+1 月因子解释 t+1 月收益（同期风险调整）
#   2. 分组方法：qcut 失败时降级为按中位数分两组，而非强制用 rank

In [99]:
import time
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

## 7.1 按HECS分组（五分位）

### 7.1.1 计算未来收益（t+1月）

In [101]:
df['Excess_Return_future'] = df.groupby('Stkcd')['Excess_Return'].shift(-1)

### 7.1.2 关键修正：因子也同步前移一个月（t+1月因子解释t+1月收益）

In [102]:
# 方法：在月份层面直接整体平移
# 先提取因子表（去重后的月度因子）
factors_original = df[['year_month', 'MKT', 'SMB', 'HML']].drop_duplicates().copy()

In [103]:
# 将因子月份向后推一个月（即 t 月因子 → t+1 月因子）
factors_original['year_month_dt'] = pd.to_datetime(factors_original['year_month'] + '-01')
factors_original['year_month_dt_shifted'] = factors_original['year_month_dt'] + pd.DateOffset(months=1)
factors_original['year_month_shifted'] = factors_original['year_month_dt_shifted'].dt.to_period('M').astype(str)

In [104]:
# 创建映射：原月份 → 移位后的因子值
factor_map = factors_original[['year_month_shifted', 'MKT', 'SMB', 'HML']].copy()
factor_map.columns = ['year_month', 'MKT_sync', 'SMB_sync', 'HML_sync']

In [105]:
print(f"  因子对齐完成：原因子从 {factors_original['year_month'].min()} 至 {factors_original['year_month'].max()}")
print(f"  移位后覆盖：{factor_map['year_month'].min()} 至 {factor_map['year_month'].max()}")

  因子对齐完成：原因子从 2011-01 至 2025-12
  移位后覆盖：2011-02 至 2026-01


### 7.1.3 只保留有 HECS 且未来收益不为空的观测

In [106]:
portfolio_df = df[df['HECS'].notna() & df['Excess_Return_future'].notna()].copy()
print(f"  有效投资组合分析样本量：{len(portfolio_df):,} 条观测")

  有效投资组合分析样本量：125,109 条观测


## 7.2 逐月按 HECS 分 5 组（P1=低HECS, P5=高HECS）

In [108]:
def assign_portfolio_robust(data):
    """
    稳健分组函数：
    1. 优先使用 pd.qcut 进行五分位等频分组
    2. 如果因重复值失败，降级为按中位数分两组（High/Low）
    """
    if len(data) < 20:
        data['Portfolio'] = np.nan
        return data
    
    # 检查 HECS 是否全相等
    if data['HECS'].nunique() == 1:
        data['Portfolio'] = np.nan
        return data
    
    try:
        # 优先：标准五分位分组
        data['Portfolio'] = pd.qcut(data['HECS'], q=5, labels=['P1', 'P2', 'P3', 'P4', 'P5'])
    except ValueError:
        # 降级：按中位数分两组（Low/High）
        median_val = data['HECS'].median()
        data['Portfolio'] = 'Low'
        data.loc[data['HECS'] > median_val, 'Portfolio'] = 'High'
        # 统一标记为 P_Low 和 P_High，以便后续处理
        data['Portfolio'] = data['Portfolio'].map({'Low': 'P1', 'High': 'P5'})
        # 记录该月使用了降级分组（用于论文注释）
        data['_group_method'] = 'median_split'
    else:
        data['_group_method'] = 'quintile'
    
    return data

In [109]:
# 按月份分组应用
portfolio_df = portfolio_df.groupby('year_month', group_keys=False).apply(assign_portfolio_robust)

In [110]:
# 剔除未能分组的观测
portfolio_df = portfolio_df[portfolio_df['Portfolio'].notna()]
print(f"  分组完成，有效观测数：{len(portfolio_df):,}")

  分组完成，有效观测数：125,109


In [111]:
# 统计分组方法的使用情况
method_counts = portfolio_df.groupby('year_month')['_group_method'].first().value_counts()
print(f"  分组方法统计：五分位 {method_counts.get('quintile', 0)} 个月，中位数降级 {method_counts.get('median_split', 0)} 个月")

  分组方法统计：五分位 142 个月，中位数降级 0 个月


In [112]:
# 清理临时列
portfolio_df = portfolio_df.drop(columns=['_group_method'], errors='ignore')

In [115]:
print("7.3 执行前，portfolio_df 的列：", portfolio_df.columns.tolist())
print("'Excess_Return_future' 是否在列中：", 'Excess_Return_future' in portfolio_df.columns)

7.3 执行前，portfolio_df 的列： ['Stkcd', 'Trdmnt', 'Mnshrtrd', 'Msmvosd', 'Mretwd', 'Msmvttl', 'Mnvaltrd', 'year_month', 'rf', 'Size', 'Excess_Return', 'MOM', 'REV', 'VOL', 'ILLIQ', 'TURN', 'n_analysts', 'ROA', 'ROE', 'GP', 'LEV', 'ATO', 'BM', 'TotalAsset', 'IA', 'ACC', 'EP', 'MKT', 'SMB', 'HML', 'RMW', 'CMA', 'in_HS300', 'in_ZZ500', 'HECS', 'Excess_Return_future', 'Portfolio']
'Excess_Return_future' 是否在列中： True


## 7.3 计算多空对冲组合收益

In [116]:
def calc_portfolio_returns(group):
    """
    对每个 (year_month, Portfolio) 组合计算等权和市值加权收益
    输入：group 是该组合下所有股票的 DataFrame
    输出：一个 Series，包含 ew_return 和 vw_return
    """
    # 等权平均
    ew = group['Excess_Return_future'].mean()
    
    # 市值加权：使用 Msmvttl 作为权重
    total_mv = group['Msmvttl'].sum()
    if total_mv == 0:
        vw = np.nan
    else:
        vw = (group['Excess_Return_future'] * group['Msmvttl']).sum() / total_mv
    
    return pd.Series({
        'ew_return': ew,
        'vw_return': vw,
        'n_stocks': len(group)  # 记录组合内股票数量
    })

In [117]:
# 按 (月份, 组合) 分组，应用计算函数
portfolio_returns = portfolio_df.groupby(['year_month', 'Portfolio']).apply(calc_portfolio_returns).reset_index()

In [118]:
print(f"  ✅ 组合收益计算完成，共 {len(portfolio_returns)} 行")
print(portfolio_returns.head())

  ✅ 组合收益计算完成，共 710 行
  year_month Portfolio  ew_return  vw_return  n_stocks
0    2014-02        P1  -0.023080  -0.008835     147.0
1    2014-02        P2  -0.006868  -0.010731     147.0
2    2014-02        P3  -0.015335  -0.014865     146.0
3    2014-02        P4  -0.040207  -0.051492     147.0
4    2014-02        P5  -0.056592  -0.054435     147.0


## 7.4 透视表

In [119]:
ew_pivot = portfolio_returns.pivot(index='year_month', columns='Portfolio', values='ew_return')
vw_pivot = portfolio_returns.pivot(index='year_month', columns='Portfolio', values='vw_return')

In [120]:
# 检查组合列
print("  P1 到 P5 列是否存在：", all([p in ew_pivot.columns for p in ['P1','P2','P3','P4','P5']]))

  P1 到 P5 列是否存在： True


In [121]:
# 多空收益
ew_pivot['long_short'] = ew_pivot['P1'] - ew_pivot['P5']
vw_pivot['long_short'] = vw_pivot['P1'] - vw_pivot['P5']

## 7.5 合并同步因子

In [122]:
factors_original = df[['year_month', 'MKT', 'SMB', 'HML']].drop_duplicates().copy()
factors_original['year_month_dt'] = pd.to_datetime(factors_original['year_month'] + '-01')
factors_original['year_month_dt_shifted'] = factors_original['year_month_dt'] + pd.DateOffset(months=1)
factors_original['year_month_shifted'] = factors_original['year_month_dt_shifted'].dt.to_period('M').astype(str)
factor_map = factors_original[['year_month_shifted', 'MKT', 'SMB', 'HML']].copy()
factor_map.columns = ['year_month', 'MKT_sync', 'SMB_sync', 'HML_sync']

In [123]:
ew_pivot = ew_pivot.reset_index().merge(factor_map, on='year_month', how='left').set_index('year_month')
vw_pivot = vw_pivot.reset_index().merge(factor_map, on='year_month', how='left').set_index('year_month')

In [124]:
print(f"  因子合并完成，多空收益有效月份数：{ew_pivot['long_short'].notna().sum()}")

  因子合并完成，多空收益有效月份数：142


## 7.6 计算 Alpha

In [126]:
from sklearn.linear_model import LinearRegression

In [127]:
def calc_performance(pivot_df, name):
    ls = pivot_df['long_short'].dropna()
    if len(ls) < 12:
        return None
    mean_ret = ls.mean() * 100
    t_stat = ls.mean() / (ls.std() / np.sqrt(len(ls)))
    
    X = pivot_df[['MKT_sync', 'SMB_sync', 'HML_sync']].dropna()
    y = pivot_df['long_short'].dropna()
    common = y.index.intersection(X.index)
    if len(common) < 12:
        alpha = np.nan
    else:
        model = LinearRegression().fit(X.loc[common], y.loc[common])
        alpha = model.intercept_ * 100
    
    return {
        '组合': name,
        '月均收益(%)': mean_ret,
        '收益t值': t_stat,
        '月均Alpha(%)': alpha,
        '年化Alpha(%)': alpha * 12 if not np.isnan(alpha) else np.nan,
        '样本月数': len(ls)
    }

In [128]:
ew_perf = calc_performance(ew_pivot, '等权多空 (P1-P5)')
vw_perf = calc_performance(vw_pivot, '市值加权多空 (P1-P5)')

In [129]:
print("\n" + "="*70)
print("【表1：HECS多空组合表现汇总】")
print("="*70)
results_df = pd.DataFrame([ew_perf, vw_perf])
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.2f}' if pd.notna(x) else 'NaN'))


【表1：HECS多空组合表现汇总】
            组合  月均收益(%)  收益t值  月均Alpha(%)  年化Alpha(%)  样本月数
  等权多空 (P1-P5)    -0.06 -0.18       -0.09       -1.05   142
市值加权多空 (P1-P5)     0.28  0.63        0.29        3.44   142


In [130]:
# 保存
ew_pivot[['P1', 'P2', 'P3', 'P4', 'P5', 'long_short']].to_csv('portfolio_returns_ew_HECS.csv')
vw_pivot[['P1', 'P2', 'P3', 'P4', 'P5', 'long_short']].to_csv('portfolio_returns_vw_HECS.csv')
print("\n  💾 已保存组合收益序列")


  💾 已保存组合收益序列


# 8. Fama-MacBeth横截面回归（模型2）

In [ ]:
# 目的：在控制多个公司特征后，检验HECS是否仍有独立的定价能力
# 方法：逐月横截面回归 → 时间序列均值 → Newey-West调整t值

In [131]:
from sklearn.linear_model import LinearRegression
from scipy.stats import t
import warnings
warnings.filterwarnings('ignore')

## 8.1 准备Fama-MacBeth回归数据

In [133]:
# 因变量：t+1月收益（已在第7节计算）
# 自变量：HECS + 控制变量
# 控制变量列表（论文中常用的定价因子）
control_vars = ['Size', 'BM', 'MOM', 'REV', 'TURN', 'VOL']

In [134]:
# 确保所有变量都存在
missing_vars = [v for v in ['HECS'] + control_vars if v not in df.columns]
if missing_vars:
    raise ValueError(f"以下变量缺失：{missing_vars}")

In [135]:
# 构建回归样本
reg_df = df[['Stkcd', 'year_month', 'Excess_Return_future', 'HECS'] + control_vars].copy()
reg_df = reg_df.dropna()
print(f"  回归样本量：{len(reg_df):,} 条观测")
print(f"  月份数：{reg_df['year_month'].nunique()}")

  回归样本量：125,109 条观测
  月份数：142


In [136]:
# HECS与各控制变量的相关系数诊断矩阵
corr_matrix = reg_df[['HECS'] + control_vars].corr()
hecs_corr = corr_matrix['HECS'].drop('HECS').sort_values(ascending=False)

print("  HECS 与各控制变量的相关系数（按月全样本计算）：")
print(hecs_corr.round(4))

  HECS 与各控制变量的相关系数（按月全样本计算）：
VOL     0.0302
MOM     0.0185
TURN    0.0061
REV     0.0050
Size   -0.0098
BM     -0.0126
Name: HECS, dtype: float64


In [137]:
# 检查是否存在高度相关（>0.7 警惕，>0.8 严重）
high_corr = hecs_corr[abs(hecs_corr) > 0.7]
if len(high_corr) > 0:
    print(f"  ⚠️ 警告：HECS 与以下变量相关性较高 (>0.7)：")
    print(high_corr.round(4))
else:
    print("  ✅ HECS 与各控制变量相关性均在 0.7 以下，无严重共线性问题")

  ✅ HECS 与各控制变量相关性均在 0.7 以下，无严重共线性问题


In [138]:
# 保存相关系数表（论文附表）
hecs_corr.to_csv('hecs_correlation_diagnosis.csv')
print("  💾 已保存相关系数诊断：hecs_correlation_diagnosis.csv")

  💾 已保存相关系数诊断：hecs_correlation_diagnosis.csv


## 8.2 定义单月横截面回归函数

In [139]:
def fm_regression_monthly(data, vars_to_keep=None):
    """
    对单个月份执行横截面回归
    因变量：Excess_Return_future（t+1月收益）
    自变量：HECS + Size + BM + MOM + REV + TURN + VOL
    """
    if vars_to_keep is None:
        vars_to_keep = ['HECS'] + control_vars
    
    X_cols = vars_to_keep
    y_col = 'Excess_Return_future'
    
    X = data[X_cols].values
    y = data[y_col].values
    n_obs = len(y)
    
    if n_obs < 30:
        return None
    
    model = LinearRegression()
    model.fit(X, y)
    
    # 计算标准误
    y_pred = model.predict(X)
    residuals = y - y_pred
    mse = np.sum(residuals**2) / (n_obs - len(X_cols) - 1)
    
    X_with_const = np.c_[np.ones(n_obs), X]
    try:
        xtx_inv = np.linalg.inv(X_with_const.T @ X_with_const)
        se = np.sqrt(mse * np.diag(xtx_inv))
    except:
        se = np.full(len(X_cols) + 1, np.nan)
    
    result = {
        'year_month': data['year_month'].iloc[0],
        'n_obs': n_obs,
        'r2': model.score(X, y)
    }
    
    result['intercept'] = model.intercept_
    result['intercept_se'] = se[0] if not np.isnan(se[0]) else np.nan
    result['intercept_t'] = model.intercept_ / se[0] if se[0] > 0 else np.nan
    
    for i, col in enumerate(X_cols):
        coef = model.coef_[i]
        se_i = se[i+1] if i+1 < len(se) else np.nan
        result[f'{col}_coef'] = coef
        result[f'{col}_se'] = se_i
        result[f'{col}_t'] = coef / se_i if se_i > 0 else np.nan
    
    return result

## 8.3 逐月执行回归

In [140]:
fm_results = []
months = sorted(reg_df['year_month'].unique())
print(f"  共 {len(months)} 个月份")

  共 142 个月份


In [141]:
for i, month in enumerate(months):
    month_data = reg_df[reg_df['year_month'] == month].copy()
    res = fm_regression_monthly(month_data)
    if res is not None:
        fm_results.append(res)
    
    if (i+1) % 24 == 0:
        print(f"  已处理 {i+1}/{len(months)} 个月份")

  已处理 24/142 个月份
  已处理 48/142 个月份
  已处理 72/142 个月份
  已处理 96/142 个月份
  已处理 120/142 个月份


In [142]:
fm_df = pd.DataFrame(fm_results)
print(f"  ✅ 成功完成 {len(fm_df)} 个月的横截面回归")
print(f"  平均样本量：{fm_df['n_obs'].mean():.0f}")
print(f"  平均R²：{fm_df['r2'].mean():.4f}")

  ✅ 成功完成 142 个月的横截面回归
  平均样本量：881
  平均R²：0.1099


## 8.4 计算时间序列均值与Newey-West标准误

In [143]:
# 修正：自动选择滞后阶数（Newey & West, 1994）
def compute_newey_lag(T):
    """
    根据 Newey & West (1994) 建议的滞后阶数选择公式
    lag = floor(4 * (T/100)^(2/9))
    其中 T 为时间序列长度
    """
    return int(np.floor(4 * (T / 100) ** (2/9)))

def newey_west_se(series, max_lag=None):
    """
    计算Newey-West调整后的标准误
    使用Bartlett权重：w(j) = 1 - j/(max_lag+1)
    """
    n = len(series)
    if n < 2:
        return np.nan
    
    # 自动选择滞后阶数
    if max_lag is None:
        max_lag = compute_newey_lag(n)
    
    # 限制最大滞后不超过 n-1
    max_lag = min(max_lag, n-1)
    
    mean_val = series.mean()
    variance = np.sum((series - mean_val)**2) / n
    
    for lag in range(1, max_lag + 1):
        autocov = np.sum((series[:-lag] - mean_val) * (series[lag:] - mean_val)) / n
        weight = 1 - lag / (max_lag + 1)
        variance += 2 * weight * autocov
    
    return np.sqrt(variance / n)

def compute_fm_summary(fm_df, var_name):
    """计算单个变量的Fama-MacBeth汇总统计"""
    coef_col = f'{var_name}_coef'
    
    if coef_col not in fm_df.columns:
        return None
    
    series = fm_df[coef_col].dropna()
    if len(series) < 2:
        return None
    
    n_obs = len(series)
    mean_coef = series.mean()
    
    # Newey-West标准误（自动选择滞后阶数）
    se_newey = newey_west_se(series)
    t_newey = mean_coef / se_newey if se_newey > 0 else np.nan
    
    # OLS标准误（用于对比）
    std_ols = series.std()
    t_ols = mean_coef / (std_ols / np.sqrt(n_obs)) if std_ols > 0 else np.nan
    
    # p值
    p_newey = 2 * (1 - t.cdf(abs(t_newey), n_obs-1)) if not np.isnan(t_newey) else np.nan
    
    # 滞后阶数信息
    lag_used = compute_newey_lag(n_obs)
    
    return {
        'Variable': var_name,
        'Mean': mean_coef,
        'Std': std_ols,
        'SE_Newey': se_newey,
        't_Newey': t_newey,
        'p_Newey': p_newey,
        'n_months': n_obs,
        'lag_used': lag_used
    }

In [144]:
# 计算所有变量的汇总
all_vars = ['intercept'] + ['HECS'] + control_vars
summary_list = []

for var in all_vars:
    res = compute_fm_summary(fm_df, var)
    if res is not None:
        summary_list.append(res)

summary_df = pd.DataFrame(summary_list)

In [145]:
# 输出滞后阶数信息（论文方法注释）
T = len(fm_df)
lag_selected = compute_newey_lag(T)
print(f"  Newey-West滞后阶数（T={T}）：按 Newey & West (1994) 公式，lag = {lag_selected}")

  Newey-West滞后阶数（T=142）：按 Newey & West (1994) 公式，lag = 4


In [147]:
print("【表2：Fama-MacBeth横截面回归结果】")
print("因变量：t+1月超额收益 (Excess_Return_future)")
print(f"Newey-West滞后阶数：{lag_selected}")

【表2：Fama-MacBeth横截面回归结果】
因变量：t+1月超额收益 (Excess_Return_future)
Newey-West滞后阶数：4


In [156]:
output_df = summary_df.copy()
# 关键修正：系数和标准误统一乘以100
output_df['Mean_pct'] = output_df['Mean'] * 100
output_df['SE_Newey_pct'] = output_df['SE_Newey'] * 100  # ← 这行之前漏掉了！

In [157]:
# 重新计算 t 值（基于统一量级后的标准误）
output_df['t_Newey_recalc'] = output_df['Mean_pct'] / output_df['SE_Newey_pct']

In [158]:
# 重新计算 p 值
from scipy.stats import t as t_dist
output_df['p_Newey_recalc'] = 2 * (1 - t_dist.cdf(abs(output_df['t_Newey_recalc']), output_df['n_months']-1))

In [159]:
# 显著性标记
def sig_stars(p):
    if p < 0.01:
        return '***'
    elif p < 0.05:
        return '**'
    elif p < 0.10:
        return '*'
    else:
        return ''

output_df['sig'] = output_df['p_Newey_recalc'].apply(sig_stars)

In [160]:
# 选择显示列
display_df = output_df[['Variable', 'Mean_pct', 'SE_Newey_pct', 't_Newey_recalc', 'sig', 'n_months']].copy()
display_df.columns = ['变量', '系数(%)', 'Newey标准误(%)', 't值', '显著性', '月份数']

In [161]:
print(display_df.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))

  变量  系数(%)  Newey标准误(%)     t值 显著性  月份数
HECS -0.457        0.777 -0.588      142
Size -0.624        0.333 -1.873   *  142
  BM -0.031        0.339 -0.093      142
 MOM  0.506        0.303  1.668   *  142
 REV  0.043        0.248  0.173      142
TURN -0.536        0.377 -1.421      142
 VOL -0.183        0.312 -0.585      142


In [162]:
print(f"\n平均调整R²：{fm_df['r2'].mean():.4f}")
print(f"平均样本量：{fm_df['n_obs'].mean():.0f} 只股票/月")


平均调整R²：0.1099
平均样本量：881 只股票/月


In [163]:
# 保存回归结果
fm_df[['year_month', 'HECS_coef'] + [f'{v}_coef' for v in control_vars]].to_csv('fm_monthly_coefficients.csv', index=False)
print("\n  💾 已保存月度系数序列：fm_monthly_coefficients.csv")

summary_df.to_csv('fm_summary_results.csv', index=False)
print("  💾 已保存汇总结果：fm_summary_results.csv")


  💾 已保存月度系数序列：fm_monthly_coefficients.csv
  💾 已保存汇总结果：fm_summary_results.csv


In [164]:
# 8.7 简要结论与行业固定效应说明（留待第10节）
hecs_row = summary_df[summary_df['Variable'] == 'HECS']
if len(hecs_row) > 0:
    hecs_t = hecs_row['t_Newey'].values[0]
    hecs_p = hecs_row['p_Newey'].values[0]
    hecs_coef = hecs_row['Mean'].values[0] * 100
    
    print(f"  HECS系数：{hecs_coef:.3f}% (t={hecs_t:.3f}, p={hecs_p:.4f})")
    if hecs_p < 0.05:
        print("  ✅ HECS在5%水平上显著，具有独立的定价能力")
    elif hecs_p < 0.10:
        print("  ⚠️ HECS在10%水平上边际显著，需要进一步检验")
    else:
        print("  ❌ HECS不显著，暂不支持假说1的强版本")

  HECS系数：-0.457% (t=-0.588, p=0.5572)
  ❌ HECS不显著，暂不支持假说1的强版本


In [165]:
print("\n  控制变量表现：")
for var in control_vars:
    row = summary_df[summary_df['Variable'] == var]
    if len(row) > 0:
        coef = row['Mean'].values[0] * 100
        t_val = row['t_Newey'].values[0]
        p_val = row['p_Newey'].values[0]
        sig = '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.10 else ''))
        print(f"    {var}: {coef:.3f}% (t={t_val:.3f}) {sig}")


  控制变量表现：
    Size: -0.624% (t=-1.873) *
    BM: -0.031% (t=-0.093) 
    MOM: 0.506% (t=1.668) *
    REV: 0.043% (t=0.173) 
    TURN: -0.536% (t=-1.421) 
    VOL: -0.183% (t=-0.585) 


# 9. 机制检验（模型3）

In [166]:
# 核心逻辑：
#   假说2：信息不对称程度越高，HECS的定价效应越强
#   假说3：模型差异程度越高，HECS的定价效应越强
# 检验方法：按分组变量将样本分为高/低两组，分别执行Fama-MacBeth回归
#   如果HECS在高组显著为负，而在低组不显著或不明显，则机制得到验证
from sklearn.linear_model import LinearRegression
from scipy.stats import t as t_dist
import warnings
warnings.filterwarnings('ignore')

## 9.1 准备分组变量

### 9.1.1 信息不对称的代理变量

In [167]:
# 代理1：分析师覆盖数量 (n_analysts)
# 1.低信息不对称：n_analysts >= 3
# 2.高信息不对称：n_analysts < 3
# 代理2：是否沪深300成分股
# 1.低信息不对称：in_HS300 == 1
# 2.高信息不对称：in_HS300 == 0 且 in_ZZ500 == 0（非沪深300也非中证500）
# 注意：由于你的样本本来就是HS300+ZZ500，这里的"高"实际上是"非HS300"（即ZZ500）这其实是一个相对较弱的信息不对称分组，因为ZZ500也有分析师覆盖

In [168]:
print("  信息不对称代理1：分析师覆盖数量 (n_analysts)")
print("    低信息不对称：>= 3 家分析师")
print("    高信息不对称：< 3 家分析师")

  信息不对称代理1：分析师覆盖数量 (n_analysts)
    低信息不对称：>= 3 家分析师
    高信息不对称：< 3 家分析师


In [169]:
print("\n  信息不对称代理2：沪深300成分股")
print("    低信息不对称：沪深300成分股")
print("    高信息不对称：非沪深300（即中证500）")


  信息不对称代理2：沪深300成分股
    低信息不对称：沪深300成分股
    高信息不对称：非沪深300（即中证500）


### 9.1.2 模型差异的代理变量

In [170]:
# 模型差异：用历史收益的截面离散度来代理
# 每个月计算全部股票超额收益的截面标准差
# 高于中位数 → 高模型差异；低于中位数 → 低模型差异

In [171]:
# 计算每月的截面离散度
monthly_dispersion = df.groupby('year_month')['Excess_Return'].std().reset_index()
monthly_dispersion.columns = ['year_month', 'cross_section_dispersion']

In [172]:
# 计算中位数
median_dispersion = monthly_dispersion['cross_section_dispersion'].median()
print(f"\n  模型差异代理：收益截面离散度")
print(f"    中位数：{median_dispersion:.4f}")
print(f"    高模型差异：> 中位数")
print(f"    低模型差异：≤ 中位数")


  模型差异代理：收益截面离散度
    中位数：0.1210
    高模型差异：> 中位数
    低模型差异：≤ 中位数


In [173]:
# 合并分组变量到主数据（但注意：回归时需要逐月分组，所以这些是全局标记）
# 对于回归样本，需要对每个月的观测打上当月的高/低标记
df = df.merge(monthly_dispersion, on='year_month', how='left')

In [174]:
# 创建分组标签
df['info_asym_group'] = 'low_info'  # 默认低信息不对称
df.loc[df['n_analysts'] < 3, 'info_asym_group'] = 'high_info'

In [175]:
df['model_div_group'] = 'low_div'
df.loc[df['cross_section_dispersion'] > median_dispersion, 'model_div_group'] = 'high_div'

In [176]:
# 为沪深300分组
df['hs300_group'] = 'non_hs300'
df.loc[df['in_HS300'] == 1, 'hs300_group'] = 'hs300'

In [177]:
print("\n  分组统计（全样本）：")
print(f"    信息不对称（分析师）：高 {df[df['info_asym_group']=='high_info']['Stkcd'].nunique()} 只，低 {df[df['info_asym_group']=='low_info']['Stkcd'].nunique()} 只")
print(f"    沪深300：{df[df['hs300_group']=='hs300']['Stkcd'].nunique()} 只，非沪深300：{df[df['hs300_group']=='non_hs300']['Stkcd'].nunique()} 只")
print(f"    模型差异：高 {df[df['model_div_group']=='high_div']['Stkcd'].nunique()} 只，低 {df[df['model_div_group']=='low_div']['Stkcd'].nunique()} 只")


  分组统计（全样本）：
    信息不对称（分析师）：高 5712 只，低 3658 只
    沪深300：700 只，非沪深300：5704 只
    模型差异：高 5710 只，低 5694 只


## 9.2 定义分组回归函数

In [178]:
def run_fm_grouped(df_full, group_col, group_name, control_vars):
    """
    对某一组的子样本执行Fama-MacBeth回归
    返回：该组的系数均值、Newey-West标准误、t值、样本月份数
    """
    # 筛选子样本
    sub_df = df_full[df_full[group_col] == group_name].copy()
    
    if len(sub_df) < 1000:
        print(f"    ⚠️ 组 {group_name} 样本量不足 ({len(sub_df)})，跳过")
        return None
    
    # 准备回归数据
    reg_sub = sub_df[['Stkcd', 'year_month', 'Excess_Return_future', 'HECS'] + control_vars].dropna()
    
    if len(reg_sub) < 500:
        print(f"    ⚠️ 组 {group_name} 回归样本量不足 ({len(reg_sub)})，跳过")
        return None
    
    months = sorted(reg_sub['year_month'].unique())
    if len(months) < 12:
        print(f"    ⚠️ 组 {group_name} 月份数不足 ({len(months)})，跳过")
        return None
    
    # 逐月回归
    results = []
    for month in months:
        month_data = reg_sub[reg_sub['year_month'] == month].copy()
        
        X_cols = ['HECS'] + control_vars
        X = month_data[X_cols].values
        y = month_data['Excess_Return_future'].values
        n_obs = len(y)
        
        if n_obs < 20:
            continue
        
        model = LinearRegression()
        model.fit(X, y)
        
        # 计算标准误（用于统计诊断，但Fama-MacBeth的最终标准误用Newey-West）
        result = {'year_month': month, 'n_obs': n_obs}
        result['intercept'] = model.intercept_
        for i, col in enumerate(X_cols):
            result[f'{col}_coef'] = model.coef_[i]
        
        results.append(result)
    
    if len(results) < 12:
        return None
    
    fm_sub = pd.DataFrame(results)
    
    # 计算每个变量的Fama-MacBeth统计量（含Newey-West）
    summary = {}
    for var in ['intercept'] + ['HECS'] + control_vars:
        if var == 'intercept':
            coef_col = 'intercept'
        else:
            coef_col = f'{var}_coef'
        
        if coef_col not in fm_sub.columns:
            continue
        
        series = fm_sub[coef_col].dropna()
        if len(series) < 2:
            continue
        
        mean_coef = series.mean()
        # Newey-West标准误（滞后4阶）
        se_newey = newey_west_se(series, max_lag=4)
        t_val = mean_coef / se_newey if se_newey > 0 else np.nan
        p_val = 2 * (1 - t_dist.cdf(abs(t_val), len(series)-1)) if not np.isnan(t_val) else np.nan
        
        summary[var] = {
            'mean': mean_coef,
            'se': se_newey,
            't': t_val,
            'p': p_val,
            'n_months': len(series)
        }
    
    return {
        'group_name': group_name,
        'n_months': len(fm_sub),
        'avg_obs': fm_sub['n_obs'].mean(),
        'summary': summary
    }

In [179]:
def newey_west_se(series, max_lag=4):
    """计算Newey-West调整后的标准误（简化版）"""
    n = len(series)
    if n < 2:
        return np.nan
    
    max_lag = min(max_lag, n-1)
    mean_val = series.mean()
    variance = np.sum((series - mean_val)**2) / n
    
    for lag in range(1, max_lag + 1):
        autocov = np.sum((series[:-lag] - mean_val) * (series[lag:] - mean_val)) / n
        weight = 1 - lag / (max_lag + 1)
        variance += 2 * weight * autocov
    
    return np.sqrt(variance / n)

## 9.3 执行分组回归：按信息不对称（分析师覆盖）

In [180]:
# 控制变量（与第8节一致）
control_vars = ['Size', 'BM', 'MOM', 'REV', 'TURN', 'VOL']

In [181]:
# 高信息不对称组：n_analysts < 3
# 低信息不对称组：n_analysts >= 3
results_info = {}
for group in ['high_info', 'low_info']:
    label = '高信息不对称' if group == 'high_info' else '低信息不对称'
    print(f"  正在运行 {label} 组...")
    res = run_fm_grouped(df, 'info_asym_group', group, control_vars)
    if res:
        results_info[group] = res
        print(f"    ✅ {label} 组完成，{res['n_months']} 个月，平均 {res['avg_obs']:.0f} 只股票/月")

  正在运行 高信息不对称 组...
    ✅ 高信息不对称 组完成，142 个月，平均 666 只股票/月
  正在运行 低信息不对称 组...
    ✅ 低信息不对称 组完成，142 个月，平均 215 只股票/月


## 9.4 执行分组回归：按信息不对称（沪深300）

In [182]:
results_hs300 = {}
for group in ['hs300', 'non_hs300']:
    label = '沪深300' if group == 'hs300' else '非沪深300（中证500）'
    print(f"  正在运行 {label} 组...")
    res = run_fm_grouped(df, 'hs300_group', group, control_vars)
    if res:
        results_hs300[group] = res
        print(f"    ✅ {label} 组完成，{res['n_months']} 个月，平均 {res['avg_obs']:.0f} 只股票/月")

  正在运行 沪深300 组...
    ✅ 沪深300 组完成，142 个月，平均 183 只股票/月
  正在运行 非沪深300（中证500） 组...
    ✅ 非沪深300（中证500） 组完成，142 个月，平均 698 只股票/月


## 9.5 执行分组回归：按模型差异

In [183]:
results_div = {}
for group in ['high_div', 'low_div']:
    label = '高模型差异' if group == 'high_div' else '低模型差异'
    print(f"  正在运行 {label} 组...")
    res = run_fm_grouped(df, 'model_div_group', group, control_vars)
    if res:
        results_div[group] = res
        print(f"    ✅ {label} 组完成，{res['n_months']} 个月，平均 {res['avg_obs']:.0f} 只股票/月")

  正在运行 高模型差异 组...
    ✅ 高模型差异 组完成，80 个月，平均 864 只股票/月
  正在运行 低模型差异 组...
    ✅ 低模型差异 组完成，62 个月，平均 903 只股票/月


## 9.6 执行四象限分组（信息不对称 × 模型差异）

In [184]:
# 创建四象限分组
df['quadrant'] = '其他'
df.loc[(df['info_asym_group'] == 'high_info') & (df['model_div_group'] == 'high_div'), 'quadrant'] = 'Q1_高信息_高差异'
df.loc[(df['info_asym_group'] == 'high_info') & (df['model_div_group'] == 'low_div'), 'quadrant'] = 'Q2_高信息_低差异'
df.loc[(df['info_asym_group'] == 'low_info') & (df['model_div_group'] == 'high_div'), 'quadrant'] = 'Q3_低信息_高差异'
df.loc[(df['info_asym_group'] == 'low_info') & (df['model_div_group'] == 'low_div'), 'quadrant'] = 'Q4_低信息_低差异'

In [185]:
results_quad = {}
for q in ['Q1_高信息_高差异', 'Q2_高信息_低差异', 'Q3_低信息_高差异', 'Q4_低信息_低差异']:
    print(f"  正在运行 {q} 组...")
    res = run_fm_grouped(df, 'quadrant', q, control_vars)
    if res:
        results_quad[q] = res
        print(f"    ✅ {q} 组完成，{res['n_months']} 个月")

  正在运行 Q1_高信息_高差异 组...
    ✅ Q1_高信息_高差异 组完成，80 个月
  正在运行 Q2_高信息_低差异 组...
    ✅ Q2_高信息_低差异 组完成，62 个月
  正在运行 Q3_低信息_高差异 组...
    ✅ Q3_低信息_高差异 组完成，80 个月
  正在运行 Q4_低信息_低差异 组...
    ✅ Q4_低信息_低差异 组完成，62 个月


In [186]:
print("【表3：机制检验 —— HECS系数分组回归结果】")

def format_result(res_dict, label_dict):
    """格式化输出单个分组变量的结果"""
    rows = []
    for key, res in res_dict.items():
        label = label_dict.get(key, key)
        if res and 'HECS' in res['summary']:
            s = res['summary']['HECS']
            rows.append({
                '组别': label,
                'HECS系数(%)': s['mean'] * 100,
                'Newey标准误(%)': s['se'] * 100,
                't值': s['t'],
                'p值': s['p'],
                '月份数': s['n_months']
            })
        else:
            rows.append({
                '组别': label,
                'HECS系数(%)': np.nan,
                'Newey标准误(%)': np.nan,
                't值': np.nan,
                'p值': np.nan,
                '月份数': np.nan
            })
    return pd.DataFrame(rows)

【表3：机制检验 —— HECS系数分组回归结果】


In [187]:
# 输出信息不对称（分析师）
label_info = {'high_info': '高信息不对称', 'low_info': '低信息不对称'}
df_info = format_result(results_info, label_info)
print("\n【按信息不对称分组（分析师覆盖）】")
print(df_info.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))


【按信息不对称分组（分析师覆盖）】
    组别  HECS系数(%)  Newey标准误(%)     t值    p值  月份数
高信息不对称     -0.264        1.080 -0.244 0.807  142
低信息不对称      0.993        5.208  0.191 0.849  142


In [188]:
# 输出信息不对称（沪深300）
label_hs300 = {'hs300': '沪深300', 'non_hs300': '非沪深300'}
df_hs300 = format_result(results_hs300, label_hs300)
print("\n【按信息不对称分组（沪深300 vs 中证500）】")
print(df_hs300.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))


【按信息不对称分组（沪深300 vs 中证500）】
    组别  HECS系数(%)  Newey标准误(%)     t值    p值  月份数
 沪深300     -0.892        3.504 -0.255 0.799  142
非沪深300     -0.202        1.129 -0.179 0.858  142


In [189]:
# 输出模型差异
label_div = {'high_div': '高模型差异', 'low_div': '低模型差异'}
df_div = format_result(results_div, label_div)
print("\n【按模型差异分组】")
print(df_div.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))


【按模型差异分组】
   组别  HECS系数(%)  Newey标准误(%)     t值    p值  月份数
高模型差异     -0.423        0.853 -0.496 0.622   80
低模型差异     -0.503        1.311 -0.383 0.703   62


In [190]:
# 输出四象限
label_quad = {
    'Q1_高信息_高差异': '高信息不对称+高模型差异',
    'Q2_高信息_低差异': '高信息不对称+低模型差异',
    'Q3_低信息_高差异': '低信息不对称+高模型差异',
    'Q4_低信息_低差异': '低信息不对称+低模型差异'
}
df_quad = format_result(results_quad, label_quad)
print("\n【四象限分组】")
print(df_quad.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))


【四象限分组】
          组别  HECS系数(%)  Newey标准误(%)     t值    p值  月份数
高信息不对称+高模型差异     -0.064        1.422 -0.045 0.964   80
高信息不对称+低模型差异     -0.522        1.591 -0.328 0.744   62
低信息不对称+高模型差异      2.907        6.572  0.442 0.659   80
低信息不对称+低模型差异     -1.476        7.411 -0.199 0.843   62


## 9.7 机制检验结论

In [191]:
# 提取关键对比
def extract_hecs_t(res_dict, key):
    if key in res_dict and res_dict[key] and 'HECS' in res_dict[key]['summary']:
        return res_dict[key]['summary']['HECS']['t']
    return np.nan

In [192]:
# 信息不对称对比（分析师）
t_high_info = extract_hecs_t(results_info, 'high_info')
t_low_info = extract_hecs_t(results_info, 'low_info')

In [193]:
# 信息不对称对比（沪深300）
t_hs300 = extract_hecs_t(results_hs300, 'hs300')
t_non_hs300 = extract_hecs_t(results_hs300, 'non_hs300')

In [194]:
# 模型差异对比
t_high_div = extract_hecs_t(results_div, 'high_div')
t_low_div = extract_hecs_t(results_div, 'low_div')

In [195]:
print("\n  信息不对称效应（分析师覆盖）：")
print(f"    高信息不对称组 HECS t值 = {t_high_info:.3f}")
print(f"    低信息不对称组 HECS t值 = {t_low_info:.3f}")
if not np.isnan(t_high_info) and not np.isnan(t_low_info):
    diff = abs(t_high_info) - abs(t_low_info)
    print(f"    差异：{diff:.3f} {'（支持假说2）' if diff > 0 else '（不支持假说2）'}")


  信息不对称效应（分析师覆盖）：
    高信息不对称组 HECS t值 = -0.244
    低信息不对称组 HECS t值 = 0.191
    差异：0.054 （支持假说2）


In [196]:
print("\n  信息不对称效应（沪深300 vs 中证500）：")
print(f"    沪深300组 HECS t值 = {t_hs300:.3f}")
print(f"    非沪深300组 HECS t值 = {t_non_hs300:.3f}")
if not np.isnan(t_hs300) and not np.isnan(t_non_hs300):
    diff = abs(t_non_hs300) - abs(t_hs300)
    print(f"    差异：{diff:.3f} {'（支持假说2）' if diff > 0 else '（不支持假说2）'}")


  信息不对称效应（沪深300 vs 中证500）：
    沪深300组 HECS t值 = -0.255
    非沪深300组 HECS t值 = -0.179
    差异：-0.075 （不支持假说2）


In [197]:
print("\n  模型差异效应：")
print(f"    高模型差异组 HECS t值 = {t_high_div:.3f}")
print(f"    低模型差异组 HECS t值 = {t_low_div:.3f}")
if not np.isnan(t_high_div) and not np.isnan(t_low_div):
    diff = abs(t_high_div) - abs(t_low_div)
    print(f"    差异：{diff:.3f} {'（支持假说3）' if diff > 0 else '（不支持假说3）'}")


  模型差异效应：
    高模型差异组 HECS t值 = -0.496
    低模型差异组 HECS t值 = -0.383
    差异：0.112 （支持假说3）


#  10. 稳健性检验

In [199]:
from sklearn.linear_model import LinearRegression
from scipy.stats import t as t_dist
import warnings
warnings.filterwarnings('ignore')

In [200]:
# 依赖检测：确认第6节函数是否可用
HECS_AVAILABLE = False
try:
    # 检查第6节的函数是否在当前环境中
    calculate_HECS_optimized
    HECS_AVAILABLE = True
    print("  ✅ 第6节 HECS 构造函数已加载")
except NameError:
    print("  ⚠️ 第6节 HECS 构造函数未加载")
    print("     如需运行参数敏感性检验（10.2），请先运行第6节代码")
    print("     其他检验（10.3-10.6）不受影响")

  ✅ 第6节 HECS 构造函数已加载


In [201]:
# 检查是否有不同 alpha 的 HECS 列
available_alphas = []
for col in df.columns:
    if col.startswith('HECS_alpha'):
        available_alphas.append(col)
if available_alphas:
    print(f"  ✅ 检测到已有 HECS 列：{available_alphas}")
else:
    print("  ℹ️ 当前仅 HECS 列（alpha=0.7）")

  ℹ️ 当前仅 HECS 列（alpha=0.7）


In [202]:
# 辅助函数定义
def newey_west_se(series, max_lag=4):
    """计算 Newey-West 调整后的标准误"""
    n = len(series)
    if n < 2:
        return np.nan
    max_lag = min(max_lag, n-1)
    mean_val = series.mean()
    variance = np.sum((series - mean_val)**2) / n
    for lag in range(1, max_lag + 1):
        autocov = np.sum((series[:-lag] - mean_val) * (series[lag:] - mean_val)) / n
        weight = 1 - lag / (max_lag + 1)
        variance += 2 * weight * autocov
    return np.sqrt(variance / n)

In [203]:
def compute_newey_lag(T):
    """Newey & West (1994) 滞后阶数公式"""
    return int(np.floor(4 * (T / 100) ** (2/9)))

In [204]:
def run_fm_regression(df_input, control_vars=None):
    """
    执行 Fama-MacBeth 回归，返回 HECS 系数和统计量
    """
    if control_vars is None:
        control_vars = ['Size', 'BM', 'MOM', 'REV', 'TURN', 'VOL']
    
    # 检查必要的列
    required_cols = ['Stkcd', 'year_month', 'Excess_Return_future', 'HECS'] + control_vars
    missing = [c for c in required_cols if c not in df_input.columns]
    if missing:
        print(f"    ⚠️ 缺少列：{missing}")
        return None
    
    reg_df = df_input[required_cols].copy()
    reg_df = reg_df.dropna()
    
    if len(reg_df) < 500:
        return None
    
    months = sorted(reg_df['year_month'].unique())
    if len(months) < 12:
        return None
    
    results = []
    for month in months:
        month_data = reg_df[reg_df['year_month'] == month].copy()
        X_cols = ['HECS'] + control_vars
        X = month_data[X_cols].values
        y = month_data['Excess_Return_future'].values
        
        if len(y) < 20:
            continue
        
        model = LinearRegression()
        model.fit(X, y)
        
        result = {'year_month': month, 'n_obs': len(y)}
        for i, col in enumerate(X_cols):
            result[f'{col}_coef'] = model.coef_[i]
        results.append(result)
    
    if len(results) < 12:
        return None
    
    fm_df = pd.DataFrame(results)
    hecs_series = fm_df['HECS_coef'].dropna()
    if len(hecs_series) < 2:
        return None
    
    mean_coef = hecs_series.mean()
    lag = compute_newey_lag(len(hecs_series))
    se_newey = newey_west_se(hecs_series, max_lag=lag)
    t_val = mean_coef / se_newey if se_newey > 0 else np.nan
    p_val = 2 * (1 - t_dist.cdf(abs(t_val), len(hecs_series)-1)) if not np.isnan(t_val) else np.nan
    
    return {
        'mean': mean_coef * 100,
        'se': se_newey * 100,
        't': t_val,
        'p': p_val,
        'n_months': len(hecs_series),
        'lag_used': lag
    }

In [205]:
def judge_robustness(baseline_t, test_t, baseline_p, test_p):
    """
    判断稳健性：
    ① 系数符号是否一致
    ② 显著性结论是否一致（都显著/都不显著）
    """
    sign_same = (baseline_t * test_t) > 0
    sig_same = (baseline_p < 0.05) == (test_p < 0.05)
    
    if sign_same and sig_same:
        return '✅ 稳健'
    elif sign_same and not sig_same:
        return '⚠️ 方向一致但显著性变化'
    else:
        return '❌ 方向变化'

## 10.1 基准结果

In [206]:
baseline = {
    'mean': -0.457,
    'se': 0.777,
    't': -0.588,
    'p': 0.557,
    'n_months': 142
}
print(f"  基准 HECS 系数：{baseline['mean']:.3f}% (t={baseline['t']:.3f}, p={baseline['p']:.4f})")

  基准 HECS 系数：-0.457% (t=-0.588, p=0.5570)


## 10.2 稳健性检验1：参数敏感性（改变 α）

In [207]:
# 先检测已有的列
existing_alpha_cols = [col for col in df.columns if col.startswith('HECS_alpha')]

if not existing_alpha_cols:
    print("  ℹ️ 当前仅 alpha=0.7 的 HECS 列。如需检验参数敏感性：")
    print("     请在第6节中分别设置 alpha=0.5 和 alpha=0.9，生成 HECS_alpha5 和 HECS_alpha9 列")
    print("     然后将它们合并到主 df 中，再运行此检验")
    alpha_results = {'alpha=0.7': baseline}
else:
    alpha_results = {'alpha=0.7': baseline}
    for col in existing_alpha_cols:
        # 提取 alpha 值：HECS_alpha5 → 0.5
        alpha_val = float(col.replace('HECS_alpha', '')) / 10
        df_temp = df.copy()
        df_temp['HECS'] = df_temp[col]
        result = run_fm_regression(df_temp)
        if result:
            alpha_results[f'alpha={alpha_val:.1f}'] = result
            print(f"  alpha={alpha_val:.1f}: 系数 {result['mean']:.3f}% (t={result['t']:.3f})")

  ℹ️ 当前仅 alpha=0.7 的 HECS 列。如需检验参数敏感性：
     请在第6节中分别设置 alpha=0.5 和 alpha=0.9，生成 HECS_alpha5 和 HECS_alpha9 列
     然后将它们合并到主 df 中，再运行此检验


## 10.3 稳健性检验2：改变分组方式（十分位）

In [209]:
def portfolio_analysis_by_quantile(df_input, q=5):
    """按 HECS 分 q 组，计算多空组合收益（等权和市值加权）"""
    
    # 1. 准备数据
    df_temp = df_input[df_input['HECS'].notna() & df_input['Excess_Return_future'].notna()].copy()
    
    if len(df_temp) < 1000:
        return None
    
    # 2. 逐月分组
    def assign_group(data):
        if len(data) < 20:
            data['Portfolio'] = np.nan
            return data
        try:
            data['Portfolio'] = pd.qcut(
                data['HECS'], q=q, 
                labels=[f'P{i+1}' for i in range(q)]
            )
        except Exception:
            data['Portfolio'] = np.nan
        return data
    
    df_temp = df_temp.groupby('year_month', group_keys=False).apply(assign_group)
    df_temp = df_temp[df_temp['Portfolio'].notna()]
    
    if len(df_temp) < 1000:
        return None
    
    # 3. 修正：分别计算等权和市值加权，避免 agg 中混用自定义函数
    
    # 3.1 等权平均（使用 groupby + mean）
    ew_returns = df_temp.groupby(['year_month', 'Portfolio'])['Excess_Return_future'].mean().reset_index()
    ew_returns.columns = ['year_month', 'Portfolio', 'ew_return']
    
    # 3.2 市值加权（使用 groupby + apply，但 apply 接收 DataFrame）
    def calc_vw(group):
        """计算单个组合的市值加权收益（group 是一个 DataFrame）"""
        total_mv = group['Msmvttl'].sum()
        if total_mv == 0:
            return np.nan
        return (group['Excess_Return_future'] * group['Msmvttl']).sum() / total_mv
    
    vw_returns = df_temp.groupby(['year_month', 'Portfolio']).apply(calc_vw).reset_index()
    vw_returns.columns = ['year_month', 'Portfolio', 'vw_return']
    
    # 3.3 合并
    portfolio_returns = ew_returns.merge(vw_returns, on=['year_month', 'Portfolio'])
    
    # 4. 构建多空组合
    pivot = portfolio_returns.pivot(index='year_month', columns='Portfolio', values='vw_return')
    
    # 检查 P1 和 Pq 是否存在
    if 'P1' not in pivot.columns or f'P{q}' not in pivot.columns:
        return None
    
    pivot['long_short'] = pivot['P1'] - pivot[f'P{q}']
    ls_series = pivot['long_short'].dropna()
    
    if len(ls_series) < 12:
        return None
    
    mean_ret = ls_series.mean() * 100
    t_val = ls_series.mean() / (ls_series.std() / np.sqrt(len(ls_series)))
    
    return {'mean': mean_ret, 't': t_val, 'n_months': len(ls_series)}

# 执行
result_q5 = portfolio_analysis_by_quantile(df, q=5)
result_q10 = portfolio_analysis_by_quantile(df, q=10)

print(f"  五分位多空收益：{result_q5['mean']:.3f}% (t={result_q5['t']:.3f})" if result_q5 else "  五分位：无法计算")
print(f"  十分位多空收益：{result_q10['mean']:.3f}% (t={result_q10['t']:.3f})" if result_q10 else "  十分位：无法计算")

if result_q5 and result_q10:
    print(f"  差异：{abs(result_q10['t'] - result_q5['t']):.3f}")

  五分位多空收益：0.278% (t=0.630)
  十分位多空收益：0.220% (t=0.405)
  差异：0.225


## 10.4 稳健性检验3：剔除极端月份

In [210]:
crisis_months = ['2015-06', '2015-07', '2015-08', '2015-09']
df_no_crisis = df[~df['year_month'].isin(crisis_months)].copy()

result_no_crisis = run_fm_regression(df_no_crisis)
if result_no_crisis:
    print(f"  剔除股灾后 HECS 系数：{result_no_crisis['mean']:.3f}% (t={result_no_crisis['t']:.3f})")
else:
    print("  ⚠️ 剔除股灾后样本量不足")

  剔除股灾后 HECS 系数：-0.490% (t=-0.620)


## 10.5 稳健性检验4：替换因变量为原始收益率

In [211]:
# 创建未来原始收益
df['Mretwd_future'] = df.groupby('Stkcd')['Mretwd'].shift(-1)

# 构建回归数据
reg_df_raw = df[['Stkcd', 'year_month', 'Mretwd_future', 'HECS'] + control_vars].copy()
reg_df_raw = reg_df_raw.dropna()

def run_fm_regression_raw(df_input, control_vars=None):
    """使用原始收益率作为因变量"""
    if control_vars is None:
        control_vars = ['Size', 'BM', 'MOM', 'REV', 'TURN', 'VOL']
    
    reg_df = df_input[['Stkcd', 'year_month', 'Mretwd_future', 'HECS'] + control_vars].copy()
    reg_df = reg_df.dropna()
    
    if len(reg_df) < 500:
        return None
    
    months = sorted(reg_df['year_month'].unique())
    if len(months) < 12:
        return None
    
    results = []
    for month in months:
        month_data = reg_df[reg_df['year_month'] == month].copy()
        X_cols = ['HECS'] + control_vars
        X = month_data[X_cols].values
        y = month_data['Mretwd_future'].values
        
        if len(y) < 20:
            continue
        
        model = LinearRegression()
        model.fit(X, y)
        
        result = {'year_month': month, 'n_obs': len(y)}
        for i, col in enumerate(X_cols):
            result[f'{col}_coef'] = model.coef_[i]
        results.append(result)
    
    if len(results) < 12:
        return None
    
    fm_df = pd.DataFrame(results)
    hecs_series = fm_df['HECS_coef'].dropna()
    if len(hecs_series) < 2:
        return None
    
    mean_coef = hecs_series.mean()
    lag = compute_newey_lag(len(hecs_series))
    se_newey = newey_west_se(hecs_series, max_lag=lag)
    t_val = mean_coef / se_newey if se_newey > 0 else np.nan
    p_val = 2 * (1 - t_dist.cdf(abs(t_val), len(hecs_series)-1)) if not np.isnan(t_val) else np.nan
    
    return {
        'mean': mean_coef * 100,
        'se': se_newey * 100,
        't': t_val,
        'p': p_val,
        'n_months': len(hecs_series)
    }

result_raw = run_fm_regression_raw(df)
if result_raw:
    print(f"  原始收益率回归 HECS 系数：{result_raw['mean']:.3f}% (t={result_raw['t']:.3f})")
else:
    print("  ⚠️ 原始收益率回归失败")

  原始收益率回归 HECS 系数：-0.457% (t=-0.588)


## 10.6 稳健性检验5：行业固定效应（有条件执行）

In [212]:
result_ind = None

if 'IndustryCode' in df.columns and df['IndustryCode'].notna().sum() > 10000:
    print("  检测到行业分类数据，正在构建行业虚拟变量...")
    
    # 取行业代码前两位作为行业大类
    df['Industry'] = df['IndustryCode'].astype(str).str[:2]
    n_industries = df['Industry'].nunique()
    print(f"  行业类别数：{n_industries}")
    
    if n_industries <= 50:
        industry_dummies = pd.get_dummies(df['Industry'], prefix='ind', drop_first=True)
        df_with_ind = pd.concat([df, industry_dummies], axis=1)
        ind_dummy_cols = [col for col in df_with_ind.columns if col.startswith('ind_')]
        
        print(f"  加入 {len(ind_dummy_cols)} 个行业虚拟变量")
        
        # ★★★ 修正：使用全部行业虚拟变量，不截断 ★★★
        control_vars_with_ind = control_vars + ind_dummy_cols
        
        # 检查样本量是否足够
        if len(df_with_ind) > 10000:
            result_ind = run_fm_regression(df_with_ind, control_vars=control_vars_with_ind)
            if result_ind:
                print(f"  加入行业固定效应后 HECS 系数：{result_ind['mean']:.3f}% (t={result_ind['t']:.3f})")
            else:
                print("  ⚠️ 行业固定效应回归失败（可能由于多重共线性或样本不足）")
        else:
            print("  ⚠️ 样本量不足，跳过行业固定效应检验")
    else:
        print(f"  ⚠️ 行业类别过多（{n_industries} > 50），跳过行业固定效应检验")
        print("     建议在论文中说明行业分类过细，固定效应可能导致自由度不足")
else:
    print("  ℹ️ 未检测到 IndustryCode 或数据不足，跳过行业固定效应检验")

  ℹ️ 未检测到 IndustryCode 或数据不足，跳过行业固定效应检验


## 10.7 结论

In [213]:
print("【表4：稳健性检验结果汇总】")
summary_rows = []

# 基准
summary_rows.append({
    '检验项目': '基准结果',
    'HECS系数(%)': baseline['mean'],
    't值': baseline['t'],
    'p值': baseline['p'],
    '月份数': baseline['n_months'],
    '结论': '基准'
})

# 参数敏感性
for name, res in alpha_results.items():
    if name != 'alpha=0.7':
        summary_rows.append({
            '检验项目': f'参数敏感性({name})',
            'HECS系数(%)': res['mean'],
            't值': res['t'],
            'p值': res['p'],
            '月份数': res['n_months'],
            '结论': judge_robustness(baseline['t'], res['t'], baseline['p'], res['p'])
        })

# 十分位分组
if result_q10:
    summary_rows.append({
        '检验项目': '十分位分组(多空收益)',
        'HECS系数(%)': result_q10['mean'],
        't值': result_q10['t'],
        'p值': np.nan,
        '月份数': result_q10['n_months'],
        '结论': judge_robustness(baseline['t'], result_q10['t'], baseline['p'], 0.5)
    })

# 剔除股灾
if result_no_crisis:
    summary_rows.append({
        '检验项目': '剔除2015股灾',
        'HECS系数(%)': result_no_crisis['mean'],
        't值': result_no_crisis['t'],
        'p值': result_no_crisis['p'],
        '月份数': result_no_crisis['n_months'],
        '结论': judge_robustness(baseline['t'], result_no_crisis['t'], baseline['p'], result_no_crisis['p'])
    })

# 原始收益率
if result_raw:
    summary_rows.append({
        '检验项目': '原始收益率(非超额)',
        'HECS系数(%)': result_raw['mean'],
        't值': result_raw['t'],
        'p值': result_raw['p'],
        '月份数': result_raw['n_months'],
        '结论': judge_robustness(baseline['t'], result_raw['t'], baseline['p'], result_raw['p'])
    })

# 行业固定效应
if result_ind:
    summary_rows.append({
        '检验项目': '加入行业固定效应',
        'HECS系数(%)': result_ind['mean'],
        't值': result_ind['t'],
        'p值': result_ind['p'],
        '月份数': result_ind['n_months'],
        '结论': judge_robustness(baseline['t'], result_ind['t'], baseline['p'], result_ind['p'])
    })

summary_df = pd.DataFrame(summary_rows)

# 格式化输出
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)))

【表4：稳健性检验结果汇总】
       检验项目  HECS系数(%)     t值    p值  月份数     结论
       基准结果     -0.457 -0.588 0.557  142     基准
十分位分组(多空收益)      0.220  0.405   NaN  142 ❌ 方向变化
   剔除2015股灾     -0.490 -0.620 0.536  138   ✅ 稳健
 原始收益率(非超额)     -0.457 -0.588 0.557  142   ✅ 稳健


In [214]:
# 统计稳健项的数量
robust_count = sum(1 for row in summary_rows if row['结论'] == '✅ 稳健')
total_tests = len(summary_rows) - 1  # 排除基准

if total_tests > 0:
    print(f"  共完成 {total_tests} 项稳健性检验，其中 {robust_count} 项结果为 '稳健'")
    if robust_count == total_tests:
        print("  ✅ HECS 不显著的结论是高度稳健的：所有检验设定下均一致")
    elif robust_count >= total_tests * 0.6:
        print("  ⚠️ HECS 不显著的结论大体稳健，但有部分设定下存在变化")
    else:
        print("  ⚠️ HECS 的显著性对不同设定较为敏感，需谨慎解读")

# 列出未完成的检验
not_completed = []
if len(alpha_results) <= 1:
    not_completed.append("参数敏感性（需其他alpha的HECS列）")
if 'IndustryCode' not in df.columns:
    not_completed.append("行业固定效应（需行业分类数据）")
if not_completed:
    print(f"\n  📌 未完成的检验项：{', '.join(not_completed)}")
    print("     建议在论文中说明这些检验因数据限制未能执行")

  共完成 3 项稳健性检验，其中 2 项结果为 '稳健'
  ⚠️ HECS 不显著的结论大体稳健，但有部分设定下存在变化

  📌 未完成的检验项：参数敏感性（需其他alpha的HECS列）, 行业固定效应（需行业分类数据）
     建议在论文中说明这些检验因数据限制未能执行
